#  Preprocessing  

> This module focuses on preprocessing single-cell RNA sequencing data, ensuring compatibility and efficiency across diverse datasets. It provides essential methods for data cleaning, normalization, and integration, facilitating seamless analysis within the Allos framework.


In [ ]:
#| default_exp preprocessing

In [ ]:
#| export

from allos.readers_tests import *



In [ ]:
#| export
import scanpy as sc


def subset_common_cells( dataset1:sc.AnnData,  # First dataset to be subsetted.
                        dataset2:sc.AnnData   # Second dataset to compare with.
                        ) -> sc.AnnData:  # Subset of `dataset1` containing only cells also found in `dataset2`.
    
    "Subset `dataset1` to only include cells that are also present in `dataset2`."
    # Find common cells by intersecting the cell identifiers of both datasets
    common_cells = dataset1.obs_names.intersection(dataset2.obs_names)
    
    # Explicitly subset dataset1 to only include these common cells
    subset_dataset1 = dataset1[common_cells, :].copy()
    
    return subset_dataset1

In [ ]:
#| export 

import pandas as pd
import anndata as ad

#| export
def transfer_obs(dataset1: ad.AnnData,  # Source AnnData object with .obs metadata to transfer.
                 dataset2: ad.AnnData   # Target AnnData object to receive .obs metadata.
                 ) -> ad.AnnData:       # The modified `dataset2` with .obs from `dataset1` transferred.
    "Transfer `.obs` metadata from `dataset1` to `dataset2` one by one, while preserving the `.var` DataFrame of `dataset2`."
    
    # Ensure dataset2's .var is preserved without altering its contents
    var_dataset2 = dataset2.var.copy()
    
    # Clear current .obs in dataset2 to ensure it only contains metadata from dataset1
    dataset2.obs = pd.DataFrame(index=dataset2.obs.index)
    
    # Loop through each column in dataset1.obs and transfer it to dataset2.obs
    for column_name in dataset1.obs.columns:
        # Transfer each column individually
        dataset2.obs[column_name] = dataset1.obs[column_name].copy()

    # Reapply the saved .var to dataset2 to ensure it's preserved
    dataset2.var = var_dataset2

    return dataset2

In [ ]:
#| export
import scanpy as sc
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

def get_sot_gene_matrix(adata):
    """
    Construct a gene-level count matrix from transcript-level data.
    
    Parameters
    ----------
    adata : AnnData
        Input AnnData with transcript-level counts. Must have a column `geneId`
        in `adata.var` containing the gene ID for each transcript.
    
    Returns
    -------
    AnnData
        A new AnnData object where columns (var) represent unique genes, 
        and values are aggregated transcript counts.
    """
    # gene_ids must be something like adata.var['geneId']
    gene_ids = adata.var['geneId'].values
    
    # Find the unique gene IDs and map each transcript to one of these genes
    unique_gene_ids, inverse = np.unique(gene_ids, return_inverse=True)
    
    # Convert adata.X to a COO matrix (cheap if already sparse)
    X_coo = coo_matrix(adata.X)

    
    # Re-map the column indices: each old column index -> new gene index
    new_row = X_coo.row
    new_col = inverse[X_coo.col]
    new_data = X_coo.data
    
    # Build the new sparse matrix
    # (COO duplicates are summed when converting to CSR/CSC)
    new_coo = coo_matrix(
        (new_data, (new_row, new_col)),
        shape=(adata.n_obs, len(unique_gene_ids))
    )
    new_X = new_coo.tocsr()  # or .tocsc(), whichever you prefer
    
    # Build a new AnnData object at gene level
    adata_gene_level = sc.AnnData(
        X=new_X,
        obs=adata.obs.copy(),
        var=pd.DataFrame(index=unique_gene_ids)
    )

    adata_gene_level.var.index.names = ['geneId']

    
    return adata_gene_level


In [ ]:
#| export
def compute_transcript_abundance_pct(adata):
    """
    Compute transcript abundance percentages ("percent spliced in") for each transcript 
    within its gene. For each cell and transcript, this is computed as:
    
        transcript_pct = (transcript count / total gene count) * 100
    
    Parameters
    ----------
    adata : AnnData
        AnnData object with transcript-level counts. Must have a column 'geneId' in adata.var.
    
    Returns
    -------
    AnnData
        A new AnnData object with the same obs and var as the input, where X holds 
        transcript abundance percentages.
    """
    gene_adata = get_sot_gene_matrix(adata)
    gene_ids = adata.var['geneId'].values
    _, inverse = np.unique(gene_ids, return_inverse=True)
    
    def to_dense(matrix):
        return matrix.toarray() if hasattr(matrix, "toarray") else matrix
    
    transcript_counts = to_dense(adata.X)
    gene_counts = to_dense(gene_adata.X)
    denominator = gene_counts[:, inverse]
    
    with np.errstate(divide='ignore', invalid='ignore'):
        pct = np.divide(
            transcript_counts,
            denominator,
            out=np.zeros_like(transcript_counts, dtype=float),
            where=denominator != 0
        )
    pct *= 100
    
    new_adata = sc.AnnData(X=pct, obs=adata.obs.copy(), var=adata.var.copy())
    return new_adata


In [ ]:
#| export
def compute_whole_data_transcript_abundance(adata):
    """
    Compute transcript abundance percentages for the entire dataset by aggregating 
    transcript counts across all cells into a single composite sample.
    
    This function sums the transcript counts over all cells, constructs a new AnnData 
    with a single observation, and then computes transcript abundance percentages 
    using compute_transcript_abundance_pct.
    
    Parameters
    ----------
    adata : AnnData
        AnnData object with transcript-level counts. Must have a 'geneId' column in adata.var.
    
    Returns
    -------
    AnnData
        A new AnnData object with one observation representing the aggregated data 
        where X holds transcript abundance percentages.
    """
    import numpy as np

    # Convert adata.X to a dense array if it's sparse
    if hasattr(adata.X, "toarray"):
        counts_dense = adata.X.toarray()
    else:
        counts_dense = adata.X

    # Sum transcript counts over all cells (i.e. aggregate along axis 0)
    aggregated_counts = np.sum(counts_dense, axis=0, keepdims=True)

    # Create a new obs DataFrame for the aggregated sample.
    # Use the first observation as a template and set its index to a unique name.
    aggregated_obs = adata.obs.iloc[[0]].copy()
    aggregated_obs.index = ["All_data"]

    # Create a new AnnData object with the aggregated transcript counts.
    aggregated_adata = sc.AnnData(X=aggregated_counts, obs=aggregated_obs, var=adata.var.copy())

    # Compute transcript abundance percentages on the aggregated data.
    return compute_transcript_abundance_pct(aggregated_adata)


In [ ]:
#| export
def filter_transcripts_by_abundance(adata, threshold_pct, verbose=False):
    """
    Filter transcripts from an AnnData object based on their overall transcript abundance percentage,
    computed by aggregating transcript counts across all cells and leveraging overall gene counts.

    The overall abundance percentage for each transcript is calculated as:
    
        overall_pct = (total transcript count) / (total gene count for its gene) * 100

    This function uses the compute_whole_data_transcript_abundance function internally to leverage the gene
    mapping information (via the 'geneId' in adata.var). Transcripts with an overall abundance
    percentage below the specified threshold (threshold_pct) are filtered out.

    Parameters
    ----------
    adata : AnnData
        AnnData object with transcript-level counts. Must have a 'geneId' column in adata.var.
    threshold_pct : float
        Minimum overall transcript abundance percentage required to keep a transcript.
    verbose : bool, optional
        If True, prints the number of transcripts kept and the threshold used. Default is False.

    Returns
    -------
    AnnData
        A new AnnData object with the filtered transcript matrix.
    """
    # Use compute_whole_data_transcript_abundance to access the gene mapping in adata.var
    pct_adata = compute_whole_data_transcript_abundance(adata)
    gene_ids = pct_adata.var['geneId'].values
    # Create an inverse mapping: for each transcript, get the index of its unique gene
    _, inverse = np.unique(gene_ids, return_inverse=True)
    
    # Helper to ensure matrices are dense for summation
    def to_dense(matrix):
        return matrix.toarray() if hasattr(matrix, "toarray") else matrix

    # Compute total transcript counts by summing over all cells (axis=0)
    transcript_counts = to_dense(adata.X)
    transcript_total = np.sum(transcript_counts, axis=0)
    
    # Compute total gene counts by summing over all cells from the gene-level matrix
    gene_adata = get_sot_gene_matrix(adata)
    gene_counts = to_dense(gene_adata.X)
    gene_total = np.sum(gene_counts, axis=0)
    
    # Compute overall transcript abundance for each transcript using the corresponding gene total count
    with np.errstate(divide='ignore', invalid='ignore'):
        overall_pct = np.divide(
            transcript_total,
            gene_total[inverse],
            out=np.zeros_like(transcript_total, dtype=float),
            where=gene_total[inverse] != 0
        )
    overall_pct *= 100

    # Create a boolean mask to retain transcripts meeting the threshold
    keep_mask = overall_pct >= threshold_pct
    kept = np.sum(keep_mask)
    total = len(keep_mask)
    if verbose:
        print(f"Filtering transcripts: keeping {kept} out of {total} transcripts (threshold = {threshold_pct}%).")
    
    # Subset the original adata to keep all metadata (obsm, uns, layers, etc.)
    return adata[:, keep_mask].copy()


In [ ]:
#| export 
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Optional

#| export

In [ ]:
#| export
import math
from typing import Optional, Tuple, List
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, gaussian_kde
from matplotlib.gridspec import GridSpec

def gene_wise_correlation(
    adata_1,
    adata_2,
    label_1="Short_Reads",
    label_2="Long_Reads",
    *,
    density_hist=True,
    facet_obs=None,
    title: Optional[str] = None,
    figsize: Tuple[float, float] = (6, 6),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    _setup_scanpy_style()

    def _density_scatter(ax, x, y, s=6, cmap="viridis", alpha=0.8):
        xy = np.vstack([x, y])
        z = gaussian_kde(xy)(xy)
        idx = np.argsort(z)
        ax.scatter(x.values[idx], y.values[idx], c=z[idx], cmap=cmap,
                   s=s, alpha=alpha, linewidths=0, rasterized=True)

    def _identity_line(ax, x, y):
        max_val = float(max(x.max(), y.max()))
        ax.plot([0, max_val], [0, max_val], ls="--", color="#aaaaaa", lw=1.2, zorder=0)

    def _annotate_r(ax, corr, p_value):
        epsilon = 1e-16
        display_pval = epsilon if p_value < epsilon else p_value
        ax.text(0.05, 0.95, f"r = {corr:.3f}, p < {display_pval:.2e}",
                transform=ax.transAxes, ha="left", va="top", fontsize=9, color="#333333")

    def _add_colorbar(fig, ax):
        sm = plt.cm.ScalarMappable(cmap="viridis")
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.35, aspect=30, pad=0.01)
        cbar.set_label("Density", fontsize=8)
        cbar.set_ticks([])

    if 'transcriptId' in (adata_1.var.index.name or ''):
        adata_1_gene = get_sot_gene_matrix(adata_1)
    else:
        adata_1_gene = adata_1

    if 'transcriptId' in (adata_2.var.index.name or ''):
        adata_2_gene = get_sot_gene_matrix(adata_2)
    else:
        adata_2_gene = adata_2

    if facet_obs is None:
        counts_1 = np.array(adata_1_gene.X.sum(axis=0)).flatten()
        counts_2 = np.array(adata_2_gene.X.sum(axis=0)).flatten()
        df_1 = pd.DataFrame({label_1: counts_1, "gene_name": adata_1_gene.var_names})
        df_2 = pd.DataFrame({label_2: counts_2, "gene_name": adata_2_gene.var_names})
        merged_df = pd.merge(df_1, df_2, on="gene_name", how="inner")
        merged_df["log_x"] = np.log1p(merged_df[label_1])
        merged_df["log_y"] = np.log1p(merged_df[label_2])
        corr, p_value = pearsonr(merged_df["log_x"], merged_df["log_y"])
        plot_title = title or f"Gene-wise Correlation\n({label_1} vs {label_2})"

        if density_hist:
            g = sns.JointGrid(data=merged_df, x="log_x", y="log_y", height=figsize[0], ratio=5)
            _density_scatter(g.ax_joint, merged_df["log_x"], merged_df["log_y"])
            _identity_line(g.ax_joint, merged_df["log_x"], merged_df["log_y"])
            _annotate_r(g.ax_joint, corr, p_value)
            g.plot_marginals(sns.kdeplot, fill=True, color="#440154", alpha=0.4, linewidth=1)
            g.ax_joint.set_xlabel(f"log({label_1} + 1)", fontsize=10)
            g.ax_joint.set_ylabel(f"log({label_2} + 1)", fontsize=10)
            g.fig.suptitle(plot_title, fontsize=12, y=1.02, fontweight="bold")
            sns.despine(ax=g.ax_joint)
            sns.despine(ax=g.ax_marg_x, left=True)
            sns.despine(ax=g.ax_marg_y, bottom=True)
            _add_colorbar(g.fig, g.ax_marg_y)  # colorbar next to right marginal
            g.fig.tight_layout()
            fig = g.fig
        else:
            fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
            _density_scatter(ax, merged_df["log_x"], merged_df["log_y"])
            _identity_line(ax, merged_df["log_x"], merged_df["log_y"])
            _annotate_r(ax, corr, p_value)
            ax.set_xlabel(f"log({label_1} + 1)", fontsize=10)
            ax.set_ylabel(f"log({label_2} + 1)", fontsize=10)
            ax.set_title(plot_title, fontsize=12, fontweight="bold")
            sns.despine(ax=ax)
            _add_colorbar(fig, ax)
            fig.tight_layout()

        if save:
            fig.savefig(save, bbox_inches="tight", dpi=dpi)
        if show:
            plt.show()
        else:
            plt.close(fig)

        if return_data:
            return merged_df, fig
        return None

    # Faceted
    categories = adata_1_gene.obs[facet_obs].unique()
    big_df_list = []
    for cat in categories:
        adata_1_sub = adata_1_gene[adata_1_gene.obs[facet_obs] == cat]
        adata_2_sub = adata_2_gene[adata_2_gene.obs[facet_obs] == cat]
        if adata_1_sub.n_obs == 0 or adata_2_sub.n_obs == 0:
            continue
        df_1 = pd.DataFrame({label_1: np.array(adata_1_sub.X.sum(axis=0)).flatten(), "gene_name": adata_1_sub.var_names})
        df_2 = pd.DataFrame({label_2: np.array(adata_2_sub.X.sum(axis=0)).flatten(), "gene_name": adata_2_sub.var_names})
        mdf = pd.merge(df_1, df_2, on="gene_name", how="inner")
        if mdf.empty:
            continue
        mdf["log_x"] = np.log1p(mdf[label_1])
        mdf["log_y"] = np.log1p(mdf[label_2])
        mdf["facet"] = cat
        big_df_list.append(mdf)

    if not big_df_list:
        print(f"No overlapping genes found for '{facet_obs}'.")
        return (pd.DataFrame(), None) if return_data else None

    big_df = pd.concat(big_df_list, ignore_index=True)
    facet_values = big_df["facet"].unique()
    n_categories = len(facet_values)
    ncols = min(3, n_categories)
    nrows = math.ceil(n_categories / ncols)
    remainder = n_categories % ncols

    if nrows > 1 and remainder != 0:
        fig = plt.figure(figsize=(4 * ncols, 4 * nrows), dpi=dpi)
        gs = GridSpec(nrows, ncols * 2, figure=fig, hspace=0.45, wspace=0.35)
        axes_flat = []
        for row in range(nrows):
            cols_in_row = ncols if row < nrows - 1 else remainder
            col_offset = ncols - cols_in_row
            for col in range(cols_in_row):
                c = (col_offset + col) * 2
                axes_flat.append(fig.add_subplot(gs[row, c:c + 2]))
    else:
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                                 figsize=(4 * ncols, 4 * nrows),
                                 dpi=dpi, constrained_layout=True)
        axes_flat = np.ravel(axes) if n_categories > 1 else [axes]

    for ax_idx, (ax, cat) in enumerate(zip(axes_flat, facet_values)):
        subset_df = big_df[big_df["facet"] == cat]
        corr, p_value = pearsonr(subset_df["log_x"], subset_df["log_y"])
        _density_scatter(ax, subset_df["log_x"], subset_df["log_y"])
        _identity_line(ax, subset_df["log_x"], subset_df["log_y"])
        _annotate_r(ax, corr, p_value)
        ax.set_title(cat, fontsize=9, fontweight="bold", pad=4)
        ax.set_xlabel(f"log({label_1} + 1)" if ax_idx // ncols == nrows - 1 else "", fontsize=8)
        ax.set_ylabel(f"log({label_2} + 1)" if ax_idx % ncols == 0 else "", fontsize=8)
        ax.tick_params(labelsize=7)
        sns.despine(ax=ax)

    sm = plt.cm.ScalarMappable(cmap="viridis")
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes_flat, shrink=0.4, aspect=25, pad=0.02)
    cbar.set_label("Density", fontsize=9)
    cbar.set_ticks([])
    fig.suptitle(title or f"Gene-wise Correlation by {facet_obs}",
                 fontsize=13, fontweight="bold", x=0.45)

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)

    if return_data:
        return big_df, fig
    return None

In [ ]:
#| export
from typing import Optional, Tuple, List
def cell_umi_violin(
    adata_1,
    adata_2,
    label_1="Short_Reads",
    label_2="Long_Reads",
    *,
    facet_obs=None,
    title: str = "UMI Distribution per Cell",
    figsize: Tuple[float, float] = (5, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    _setup_scanpy_style()

    umis_1 = np.array(adata_1.X.sum(axis=1)).flatten()
    umis_2 = np.array(adata_2.X.sum(axis=1)).flatten()

    df_1 = pd.DataFrame({"log_umis": np.log1p(umis_1), "cell": adata_1.obs_names, "dataset": label_1})
    df_2 = pd.DataFrame({"log_umis": np.log1p(umis_2), "cell": adata_2.obs_names, "dataset": label_2})

    if facet_obs:
        df_1[facet_obs] = adata_1.obs[facet_obs].values
        df_2[facet_obs] = adata_2.obs[facet_obs].values

    combined = pd.concat([df_1, df_2], ignore_index=True)

    if facet_obs is None:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        sns.violinplot(
            data=combined, x="dataset", y="log_umis",
            palette=["#4C72B0", "#DD8452"],
            inner="box", linewidth=0.8, ax=ax
        )
        ax.set_xlabel("")
        ax.set_ylabel("log(UMIs + 1)", fontsize=10)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=15, ha="right", fontsize=9)
        ax.set_title(title, fontsize=12, fontweight="bold")
        sns.despine(ax=ax)
        fig.tight_layout()
    else:
        cell_types = combined[facet_obs].dropna().unique()
        n = len(cell_types)
        ncols = min(3, n)
        nrows = math.ceil(n / ncols)
        remainder = n % ncols

        if nrows > 1 and remainder != 0:
            fig = plt.figure(figsize=(figsize[0] * ncols, figsize[1] * nrows), dpi=dpi)
            gs = GridSpec(nrows, ncols * 2, figure=fig, hspace=0.5, wspace=0.4)
            axes_flat = []
            for row in range(nrows):
                cols_in_row = ncols if row < nrows - 1 else remainder
                col_offset = ncols - cols_in_row
                for col in range(cols_in_row):
                    c = (col_offset + col) * 2
                    axes_flat.append(fig.add_subplot(gs[row, c:c + 2]))
        else:
            fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                                     figsize=(figsize[0] * ncols, figsize[1] * nrows),
                                     dpi=dpi, constrained_layout=True)
            axes_flat = np.ravel(axes) if n > 1 else [axes]

        for ax_idx, (ax, ct) in enumerate(zip(axes_flat, cell_types)):
            subset = combined[combined[facet_obs] == ct]
            sns.violinplot(
                data=subset, x="dataset", y="log_umis",
                palette=["#4C72B0", "#DD8452"],
                inner="box", linewidth=0.8, ax=ax
            )
            ax.set_title(ct, fontsize=9, fontweight="bold", pad=4)
            ax.set_xlabel("")
            ax.set_ylabel("log(UMIs + 1)" if ax_idx % ncols == 0 else "", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.tick_params(axis="x", labelsize=7, rotation=15)
            sns.despine(ax=ax)

        fig.suptitle(f"{title} by {facet_obs}", fontsize=13, fontweight="bold", x=0.45)

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)

    if return_data:
        return combined, fig
    return None

In [ ]:
#| export
import math
from typing import Optional, Tuple, List
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from matplotlib.gridspec import GridSpec

def gene_wise_bland_altman(
    adata_1,
    adata_2,
    label_1="Short_Reads",
    label_2="Long_Reads",
    *,
    facet_obs=None,
    title: Optional[str] = None,
    figsize: Tuple[float, float] = (6, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    _setup_scanpy_style()

    if adata_1.var.index.name is not None and "transcriptId" in adata_1.var.index.name:
        adata_1_gene = get_sot_gene_matrix(adata_1)
    else:
        adata_1_gene = adata_1

    if adata_2.var.index.name is not None and "transcriptId" in adata_2.var.index.name:
        adata_2_gene = get_sot_gene_matrix(adata_2)
    else:
        adata_2_gene = adata_2

    def _merge_summed(adataA, adataB):
        countsA = np.array(adataA.X.sum(axis=0)).flatten()
        countsB = np.array(adataB.X.sum(axis=0)).flatten()
        dfA = pd.DataFrame({label_1: countsA, "gene_name": adataA.var_names})
        dfB = pd.DataFrame({label_2: countsB, "gene_name": adataB.var_names})
        merged = pd.merge(dfA, dfB, on="gene_name", how="inner")
        merged["log_x"] = np.log1p(merged[label_1])
        merged["log_y"] = np.log1p(merged[label_2])
        merged["mean_val"] = (merged["log_x"] + merged["log_y"]) / 2
        merged["diff_val"] = merged["log_x"] - merged["log_y"]
        return merged

    def _density_scatter(ax, x, y, s=6, cmap="viridis", alpha=0.8):
        xy = np.vstack([x, y])
        z = gaussian_kde(xy)(xy)
        idx = np.argsort(z)
        ax.scatter(x[idx], y[idx], c=z[idx], cmap=cmap, s=s,
                   alpha=alpha, linewidths=0, rasterized=True)

    def _ba_lines(ax, mean_diff, std_diff):
        loa_upper = mean_diff + 1.96 * std_diff
        loa_lower = mean_diff - 1.96 * std_diff
        ax.axhline(0, color="#cccccc", lw=0.8, zorder=0)
        ax.axhline(mean_diff, color="#333333", lw=1.2, ls="-", zorder=1)
        ax.axhline(loa_upper, color="#E05C5C", lw=1.0, ls="--", zorder=1)
        ax.axhline(loa_lower, color="#E05C5C", lw=1.0, ls="--", zorder=1)
        kw = dict(va="bottom", ha="right", fontsize=7, transform=ax.get_yaxis_transform())
        ax.text(1.0, mean_diff, f" {mean_diff:+.2f}", color="#333333", **kw)
        ax.text(1.0, loa_upper, f" {loa_upper:+.2f}", color="#E05C5C", **kw)
        ax.text(1.0, loa_lower, f" {loa_lower:+.2f}", color="#E05C5C", **kw)

    def _style_ax(ax, ax_idx, nrows, ncols):
        ax.tick_params(labelsize=7)
        ax.set_xlabel(
            f"Mean of log({label_1}, {label_2})" if ax_idx // ncols == nrows - 1 else "",
            fontsize=8
        )
        ax.set_ylabel(
            f"Difference ({label_1} − {label_2})" if ax_idx % ncols == 0 else "",
            fontsize=8
        )
        sns.despine(ax=ax)

    if facet_obs is None:
        merged_df = _merge_summed(adata_1_gene, adata_2_gene)
        mean_diff = merged_df["diff_val"].mean()
        std_diff  = merged_df["diff_val"].std(ddof=1)
        plot_title = title or f"Bland–Altman\n{label_1} vs {label_2}"

        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        _density_scatter(ax, merged_df["mean_val"].values, merged_df["diff_val"].values)
        _ba_lines(ax, mean_diff, std_diff)
        ax.set_xlabel(f"Mean of log({label_1}, {label_2})", fontsize=10)
        ax.set_ylabel(f"Difference ({label_1} − {label_2})", fontsize=10)
        ax.set_title(plot_title, fontsize=12, fontweight="bold")

        sm = plt.cm.ScalarMappable(cmap="viridis")
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.6, aspect=20, pad=0.02)
        cbar.set_label("Density", fontsize=8)
        cbar.set_ticks([])

        sns.despine(ax=ax)
        fig.tight_layout()

        if save:
            fig.savefig(save, bbox_inches="tight", dpi=dpi)
        if show:
            plt.show()
        else:
            plt.close(fig)

        if return_data:
            return merged_df, fig
        return None

    # Faceted
    categories = adata_1_gene.obs[facet_obs].unique()
    all_cats_list = []
    for cat in categories:
        a1 = adata_1_gene[adata_1_gene.obs[facet_obs] == cat]
        a2 = adata_2_gene[adata_2_gene.obs[facet_obs] == cat]
        if a1.n_obs == 0 or a2.n_obs == 0:
            continue
        mdf = _merge_summed(a1, a2)
        if mdf.empty:
            continue
        mdf["facet"] = cat
        all_cats_list.append(mdf)

    if not all_cats_list:
        print(f"No data for any categories in '{facet_obs}'.")
        return (pd.DataFrame(), None) if return_data else None

    big_df = pd.concat(all_cats_list, ignore_index=True)
    facet_values = big_df["facet"].unique()
    n_categories = len(facet_values)
    ncols = min(3, n_categories)
    nrows = math.ceil(n_categories / ncols)
    remainder = n_categories % ncols

    if nrows > 1 and remainder != 0:
        fig = plt.figure(figsize=(4 * ncols, 4 * nrows), dpi=dpi)
        gs = GridSpec(nrows, ncols * 2, figure=fig, hspace=0.45, wspace=0.45)
        axes_flat = []
        for row in range(nrows):
            cols_in_row = ncols if row < nrows - 1 else remainder
            col_offset = ncols - cols_in_row
            for col in range(cols_in_row):
                c = (col_offset + col) * 2
                axes_flat.append(fig.add_subplot(gs[row, c:c + 2]))
    else:
        fig, axes = plt.subplots(nrows=nrows, ncols=ncols,
                                 figsize=(4 * ncols, 4 * nrows),
                                 dpi=dpi, constrained_layout=True)
        axes_flat = np.ravel(axes) if n_categories > 1 else [axes]

    for ax_idx, (ax, cat) in enumerate(zip(axes_flat, facet_values)):
        subset = big_df[big_df["facet"] == cat]
        mean_diff = subset["diff_val"].mean()
        std_diff  = subset["diff_val"].std(ddof=1)
        _density_scatter(ax, subset["mean_val"].values, subset["diff_val"].values)
        _ba_lines(ax, mean_diff, std_diff)
        ax.set_title(cat, fontsize=9, fontweight="bold", pad=4)
        _style_ax(ax, ax_idx, nrows, ncols)

    sm = plt.cm.ScalarMappable(cmap="viridis")
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes_flat, shrink=0.4, aspect=25, pad=0.02)
    cbar.set_label("Density", fontsize=9)
    cbar.set_ticks([])
    fig.suptitle(title or f"Bland–Altman by {facet_obs}\n({label_1} vs {label_2})",
                 fontsize=13, fontweight="bold", x=0.45)

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)

    if return_data:
        return big_df, fig
    return None

In [ ]:
# | export
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import sparse
from typing import Optional, Union, List, Tuple, Dict

# -------------------- Scanpy-style color palettes --------------------
ALLOS_PALETTE = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"
]

ALLOS_PALETTE_PASTEL = [
    "#a6cee3", "#b2df8a", "#fb9a99", "#fdbf6f", "#cab2d6",
    "#ffff99", "#b15928", "#e31a1c", "#33a02c", "#1f78b4"
]

ISOFORM_COLORS = ["#3498db", "#2ecc71", "#f39c12", "#9b59b6"]  # blue, green, orange, purple

# -------------------- helpers --------------------
def _strip_ver(ids): 
    """Strip version numbers from transcript/gene IDs."""
    return pd.Index([str(x).split(".")[0] for x in pd.Index(ids)])

def _get_matrix(adata, layer):
    """Get the appropriate matrix from adata (layer or X)."""
    if layer is None:
        layer = "counts" if "counts" in adata.layers else None
    X = adata.layers[layer] if (layer is not None) else adata.X
    return X, layer

def _present_tx_in_group(adata, obs_mask, layer=None, min_total=1, adata_tx_col=None):
    """Return version-stripped transcript IDs present in this group."""
    X, _ = _get_matrix(adata, layer)
    Xg = X[obs_mask] if hasattr(X, "shape") else X
    s = np.asarray(Xg.sum(axis=0)).ravel() if sparse.issparse(Xg) else Xg.sum(axis=0)
    idx_present = np.where(s >= float(min_total))[0]
    if adata_tx_col is None:
        tx_ids = _strip_ver(adata.var.index[idx_present])
    else:
        tx_ids = _strip_ver(adata.var.loc[adata.var.index[idx_present], adata_tx_col].values)
    return pd.Index(tx_ids)

def _bin_counts(iso_per_gene, labels=("1","2-3","4-5","≥6")):
    """Bin isoform counts into categories."""
    binned = pd.cut(iso_per_gene, [0,1,3,5,np.inf], labels=list(labels),
                    right=True, include_lowest=True)
    counts = binned.value_counts().reindex(labels).fillna(0).astype(int)
    total = int(counts.sum())
    perc = (counts / max(total,1) * 100)
    return counts, perc, total

def _setup_scanpy_style():
    """Configure matplotlib for scanpy-like aesthetics."""
    plt.rcParams.update({
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans', 'Helvetica'],
        'font.size': 11,
        'axes.titlesize': 13,
        'axes.titleweight': 'bold',
        'axes.labelsize': 11,
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.linewidth': 1.2,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 9,
        'legend.frameon': False,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
        'savefig.dpi': 150,
        'savefig.bbox': 'tight',
    })

# -------------------- single plot --------------------
def plot_isoforms_per_gene(
    td, 
    *, 
    adata=None, 
    adata_tx_col=None,
    title: str = "Isoforms per Gene",
    colors: Optional[List] = None,
    show_percentages: bool = True,
    figsize: Tuple[float, float] = (5, 4),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot isoform distribution using TranscriptData object.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object from allos.transcript_data.
    adata : AnnData, optional
        If provided, filter to transcripts present in adata.
    adata_tx_col : str, optional
        Column in adata.var containing transcript IDs. If None, uses var.index.
    title : str
        Plot title.
    colors : list, optional
        Colors for bars. If None, uses ALLOS palette.
    show_percentages : bool
        Whether to show percentage labels on bars.
    figsize : tuple
        Figure size.
    dpi : int
        Figure DPI.
    save : str, optional
        Path to save figure.
    show : bool
        Whether to display the plot.
    return_data : bool
        If True, return (DataFrame, fig, ax). Otherwise return None.
    
    Returns
    -------
    None or tuple
        If return_data=True: (DataFrame with bin counts, figure, axes)
    """
    _setup_scanpy_style()
    
    tx = td.gene_transcript_table(use_gene_name=True)

    if adata is not None:
        ad_tx = _strip_ver(adata.var.index if adata_tx_col is None
                           else adata.var[adata_tx_col].values)
        tx = tx[tx["transcript_id"].isin(set(ad_tx))]

    iso_per_gene = tx.groupby("gene")["transcript_id"].nunique()
    labels = ["1", "2-3", "4-5", "≥6"]
    counts, perc, total = _bin_counts(iso_per_gene, labels)

    if colors is None:
        colors = ISOFORM_COLORS

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    bars = ax.bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5, width=0.7)
    
    ax.set_title(title, pad=15)
    ax.set_xlabel("Number of isoforms", labelpad=10)
    ax.set_ylabel("Number of genes", labelpad=10)
    ax.set_ylim(0, counts.max() * 1.22)
    
    # Add subtle grid
    ax.yaxis.grid(True, linestyle='--', alpha=0.3, color='gray')
    ax.set_axisbelow(True)
    
    # Percentage labels
    if show_percentages:
        for i, (bar, c, p) in enumerate(zip(bars, counts.values, perc.values)):
            height = bar.get_height()
            ax.annotate(
                f'{c:,}\n({p:.1f}%)',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', va='bottom',
                fontsize=9, fontweight='medium',
                color='#333333'
            )
    
    sns.despine(ax=ax)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return pd.DataFrame({"bin": labels, "genes": counts.values, "percent": perc.values}), fig, ax
    return None

def plot_isoforms_per_gene_by_group(
    td, 
    adata, 
    groupby, 
    *,
    layer: Optional[str] = None, 
    min_total: int = 1,
    adata_tx_col: Optional[str] = None,
    title: Optional[str] = None,
    palette: Optional[Union[List, Dict]] = None,
    figsize: Optional[Tuple[float, float]] = None,
    dpi: int = 150,
    bar_mode: str = "grouped",
    show_legend: bool = True,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot isoforms per gene by group using TranscriptData object.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object from allos.transcript_data.
    adata : AnnData
        AnnData object with transcript-level data.
    groupby : str or list
        Column(s) in adata.obs to group by.
    layer : str, optional
        Layer to use for counts. If None, uses 'counts' layer or X.
    min_total : int
        Minimum total count for a transcript to be considered present.
    adata_tx_col : str, optional
        Column in adata.var containing transcript IDs.
    title : str, optional
        Plot title. If None, auto-generated.
    palette : list or dict, optional
        Colors for groups. Pass a list (ordered by group) or a dict
        mapping {group_name: color}. If None, uses ALLOS palette.
    figsize : tuple, optional
        Figure size. If None, auto-calculated.
    dpi : int
        Figure DPI.
    bar_mode : str
        'grouped' for side-by-side bars, 'faceted' for separate subplots.
    show_legend : bool
        Whether to show legend (grouped mode only).
    save : str, optional
        Path to save figure.
    show : bool
        Whether to display the plot.
    return_data : bool
        If True, return (dict, fig). Otherwise return None.
    
    Returns
    -------
    None or tuple
        If return_data=True: (dict with counts/percent/totals, figure)
    """
    _setup_scanpy_style()
    
    tx = td.gene_transcript_table(use_gene_name=True)

    ad_tx_all = _strip_ver(adata.var.index if adata_tx_col is None
                           else adata.var[adata_tx_col].values)
    tx = tx[tx["transcript_id"].isin(set(ad_tx_all))]

    if isinstance(groupby, (list, tuple)):
        labels_series = adata.obs[groupby].astype(str).agg(" | ".join, axis=1)
        gname = " | ".join(groupby)
    else:
        labels_series = adata.obs[groupby].astype(str)
        gname = groupby
    groups = labels_series.unique()

    # Build color mapping: supports None, list, or dict
    if palette is None:
        colors = {g: ALLOS_PALETTE[i] for i, g in enumerate(groups)}
    elif isinstance(palette, dict):
        colors = palette
    else:
        colors = {g: palette[i] for i, g in enumerate(groups)}

    bins_labels = ["1", "2-3", "4-5", "≥6"]
    counts_df, perc_df, totals = [], [], []

    for g in groups:
        mask = (labels_series.values == g)
        present_tx = _present_tx_in_group(adata, mask, layer=layer, min_total=min_total, adata_tx_col=adata_tx_col)
        tx_g = tx[tx["transcript_id"].isin(present_tx)]
        iso_per_gene = tx_g.groupby("gene")["transcript_id"].nunique()
        c, p, tot = _bin_counts(iso_per_gene, bins_labels)
        counts_df.append(c.rename(g))
        perc_df.append(p.rename(g))
        totals.append((g, tot))

    counts = pd.DataFrame(counts_df).fillna(0).astype(int)
    perc = pd.DataFrame(perc_df).fillna(0.0)

    if title is None:
        title = f"Isoforms per gene by {gname}"

    # -------- FACETED MODE --------
    if bar_mode == "faceted" or len(groups) > 8:
        n = len(groups)
        if figsize is None:
            figsize = (min(3.5 * n, 20), 4)
        
        fig, axes = plt.subplots(1, n, figsize=figsize, dpi=dpi, sharey=True)
        if n == 1: 
            axes = [axes]
        
        for ax, g in zip(axes, groups):
            bars = ax.bar(bins_labels, counts.loc[g].values, 
                         color=ISOFORM_COLORS, edgecolor='white', linewidth=1, width=0.7)
            ax.set_title(str(g), fontsize=10, fontweight='bold', pad=8)
            ax.set_xlabel("Isoforms/gene", fontsize=9, labelpad=5)
            ax.set_ylim(0, counts.to_numpy().max() * 1.15)
            ax.yaxis.grid(True, linestyle='--', alpha=0.3, color='gray')
            ax.set_axisbelow(True)
            ax.tick_params(axis='x', labelsize=8)
        
        axes[0].set_ylabel("Number of genes", labelpad=8)
        fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
    
    # -------- GROUPED MODE --------
    else:
        if figsize is None:
            figsize = (max(6, len(groups) * 0.8 + 4), 5)
        
        width = 0.75 / len(groups)
        xs = np.arange(len(bins_labels))
        
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        
        for i, g in enumerate(groups):
            offset = (i - len(groups)/2 + 0.5) * width
            bars = ax.bar(xs + offset, counts.loc[g].values, 
                         width=width, label=str(g), color=colors[g],
                         edgecolor='white', linewidth=0.8)
        
        ax.set_xticks(xs)
        ax.set_xticklabels(bins_labels)
        ax.set_ylabel("Number of genes", labelpad=10)
        ax.set_xlabel("Number of isoforms", labelpad=10)
        ax.set_title(title, pad=15)
        ax.set_ylim(0, counts.to_numpy().max() * 1.18)
        
        ax.yaxis.grid(True, linestyle='--', alpha=0.3, color='gray')
        ax.set_axisbelow(True)
        
        if show_legend:
            ax.legend(
                ncol=min(len(groups), 4), 
                loc='upper right',
                fontsize=8,
                title=gname,
                title_fontsize=9,
                bbox_to_anchor=(1.0, 1.0)
            )
        
        plt.tight_layout()

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {"counts": counts, "percent": perc, "totals": dict(totals)}, fig
    return None


# -------------------- Transcript Length Analysis --------------------

def plot_transcript_length_distribution(
    td,
    adata,
    groupby: Optional[str] = None,
    *,
    adata_tx_col: Optional[str] = None,
    layer: Optional[str] = None,
    log_scale: bool = True,  # Changed default to True
    plot_type: str = "violin",
    show_stats: bool = True,  # New: show statistical annotations
    title: Optional[str] = None,
    palette: Optional[List] = None,
    figsize: Optional[Tuple[float, float]] = None,
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot distribution of transcript lengths, optionally grouped by cell type.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object.
    adata : AnnData
        AnnData object with transcript-level data.
    groupby : str, optional
        Column in adata.obs to group by.
    log_scale : bool
        Whether to log10-transform lengths (default True for better visualization).
    plot_type : str
        "violin", "box", or "ridge".
    show_stats : bool
        Whether to show statistical summary (median, IQR).
    """
    import seaborn as sns
    from scipy import stats
    _setup_scanpy_style()
    
    length_table = td.transcript_lengths_table()
    
    if adata_tx_col is None:
        adata_tx = _strip_ver(adata.var.index)
    else:
        adata_tx = _strip_ver(adata.var[adata_tx_col].values)
    
    length_table = length_table[length_table["transcript_id"].isin(set(adata_tx))].copy()
    
    if length_table.empty:
        print("No matching transcripts found.")
        return None
    
    if groupby is not None:
        X, _ = _get_matrix(adata, layer)
        tx_to_idx = {tx: i for i, tx in enumerate(adata_tx)}
        length_table["adata_idx"] = length_table["transcript_id"].map(tx_to_idx)
        length_table = length_table.dropna(subset=["adata_idx"])
        length_table["adata_idx"] = length_table["adata_idx"].astype(int)
        
        groups = adata.obs[groupby].unique()
        plot_data = []
        
        for g in groups:
            mask = (adata.obs[groupby].values == g)
            Xg = X[mask]
            expr_sum = np.asarray(Xg.sum(axis=0)).ravel() if sparse.issparse(Xg) else Xg.sum(axis=0)
            
            for _, row in length_table.iterrows():
                idx = row["adata_idx"]
                if expr_sum[idx] > 0:
                    plot_data.append({
                        "group": g,
                        "transcript_id": row["transcript_id"],
                        "length": row["length"],
                        "expression": expr_sum[idx]
                    })
        
        df = pd.DataFrame(plot_data)
    else:
        df = length_table[["transcript_id", "length"]].copy()
        df["group"] = "All"
    
    if df.empty:
        print("No expressed transcripts found.")
        return None
    
    # Store original for stats
    df["length_original"] = df["length"]
    
    if log_scale:
        df["length"] = np.log10(df["length"].clip(lower=1))
        length_label = "Transcript length (log₁₀ bp)"
    else:
        length_label = "Transcript length (bp)"
    
    n_groups = df["group"].nunique()
    if figsize is None:
        figsize = (max(6, n_groups * 0.9 + 2), 5)
    
    if palette is None:
        palette = ALLOS_PALETTE[:n_groups]
    
    if title is None:
        title = "Transcript Length Distribution"
        if groupby:
            title += f" by {groupby}"
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    if plot_type == "violin":
        sns.violinplot(data=df, x="group", y="length", palette=palette, ax=ax,
                      inner="box", linewidth=1, cut=0, density_norm="width")
    elif plot_type == "box":
        sns.boxplot(data=df, x="group", y="length", palette=palette, ax=ax,
                   linewidth=1.5, fliersize=2, notch=True)
    
    ax.set_xlabel("")
    ax.set_ylabel(length_label)
    ax.set_title(title, pad=15)
    plt.xticks(rotation=45, ha='right')
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Add statistics annotation
    if show_stats:
        median_bp = df["length_original"].median()
        q25, q75 = df["length_original"].quantile([0.25, 0.75])
        stats_text = f"Median: {median_bp:,.0f} bp\nIQR: {q25:,.0f}-{q75:,.0f} bp\nn = {len(df):,}"
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
                ha='right', va='top', fontsize=9, color='#444444',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='#cccccc'))
        
        # Add Kruskal-Wallis test if multiple groups
        if n_groups > 1 and groupby is not None:
            group_data = [df[df["group"] == g]["length_original"].values for g in df["group"].unique()]
            if all(len(g) > 0 for g in group_data):
                h_stat, p_val = stats.kruskal(*group_data)
                sig_text = f"Kruskal-Wallis p = {p_val:.2e}" if p_val < 0.001 else f"Kruskal-Wallis p = {p_val:.3f}"
                ax.text(0.02, 0.98, sig_text, transform=ax.transAxes,
                        ha='left', va='top', fontsize=8, color='#666666')
    
    # Add reference lines for typical transcript sizes
    if log_scale:
        for size, label in [(1000, "1kb"), (5000, "5kb"), (10000, "10kb")]:
            if np.log10(size) < ax.get_ylim()[1] and np.log10(size) > ax.get_ylim()[0]:
                ax.axhline(np.log10(size), color='gray', linestyle=':', alpha=0.5, linewidth=1)
                ax.text(ax.get_xlim()[1], np.log10(size), f" {label}", va='center', fontsize=7, color='gray')
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return df, fig
    return None


def plot_detected_transcript_lengths(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    log_scale: bool = True,
    show_stats: bool = True,
    bins: int = 50,
    title: str = "Detected Transcript Lengths",
    color: Optional[str] = None,
    figsize: Tuple[float, float] = (8, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Simple histogram of annotated lengths for all detected transcripts.
    
    Shows the length distribution of transcripts present in your dataset
    based on GTF annotations.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object.
    adata : AnnData
        AnnData object with transcript-level data.
    adata_tx_col : str, optional
        Column in adata.var containing transcript IDs.
    log_scale : bool
        Whether to log10-transform lengths.
    show_stats : bool
        Whether to show summary statistics.
    bins : int
        Number of histogram bins.
    """
    import seaborn as sns
    _setup_scanpy_style()
    
    length_table = td.transcript_lengths_table()
    
    if adata_tx_col is None:
        adata_tx = set(_strip_ver(adata.var.index))
    else:
        adata_tx = set(_strip_ver(adata.var[adata_tx_col].values))
    
    df = length_table[length_table["transcript_id"].isin(adata_tx)].copy()
    df = df[df["length"] > 0]
    
    if df.empty:
        print("No matching transcripts found.")
        return None
    
    if log_scale:
        df["length_plot"] = np.log10(df["length"])
        length_label = "Transcript length (log₁₀ bp)"
    else:
        df["length_plot"] = df["length"]
        length_label = "Transcript length (bp)"
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    if color is None:
        color = ALLOS_PALETTE[0]
    
    sns.histplot(df["length_plot"], bins=bins, kde=True, ax=ax,
                color=color, edgecolor='white', linewidth=0.5, alpha=0.7)
    
    ax.set_xlabel(length_label)
    ax.set_ylabel("Number of transcripts")
    ax.set_title(title, pad=15)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Add reference lines for common sizes
    if log_scale:
        for kb, label in [(500, "500bp"), (1000, "1kb"), (2000, "2kb"), (5000, "5kb"), (10000, "10kb")]:
            log_val = np.log10(kb)
            if ax.get_xlim()[0] < log_val < ax.get_xlim()[1]:
                ax.axvline(log_val, color='gray', linestyle=':', alpha=0.5, linewidth=1)
                ax.text(log_val, ax.get_ylim()[1] * 0.95, f' {label}', 
                       fontsize=8, color='gray', va='top')
    
    # Statistics box
    if show_stats:
        median_bp = df["length"].median()
        mean_bp = df["length"].mean()
        q25, q75 = df["length"].quantile([0.25, 0.75])
        
        stats_text = (f"n = {len(df):,} transcripts\n"
                     f"Median: {median_bp:,.0f} bp\n"
                     f"Mean: {mean_bp:,.0f} bp\n"
                     f"IQR: {q25:,.0f} - {q75:,.0f} bp")
        
        ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
               ha='right', va='top', fontsize=9,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='#cccccc'))
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return df, fig
    return None


def plot_biotype_breakdown(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    top_n: int = 8,
    title: str = "Transcript Biotype Breakdown",
    colors: Optional[List] = None,
    figsize: Tuple[float, float] = (7, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Bar chart showing breakdown of transcripts by biotype (protein_coding, lncRNA, etc.).
    
    Similar style to the isoforms-per-gene plot.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object.
    adata : AnnData
        AnnData object with transcript-level data.
    adata_tx_col : str, optional
        Column in adata.var containing transcript IDs.
    top_n : int
        Number of top biotypes to show (rest grouped as "other").
    """
    _setup_scanpy_style()
    
    # Get biotype info from TranscriptData
    biotype_data = []
    for tid, rec in td._idx._tx_by_id.items():
        tid_stripped = tid.split(".")[0]
        biotype = rec.transcript_type if rec.transcript_type and rec.transcript_type != "nan" else "unknown"
        biotype_data.append({"transcript_id": tid_stripped, "biotype": biotype})
    
    biotype_df = pd.DataFrame(biotype_data)
    
    # Filter to transcripts in adata
    if adata_tx_col is None:
        adata_tx = set(_strip_ver(adata.var.index))
    else:
        adata_tx = set(_strip_ver(adata.var[adata_tx_col].values))
    
    df = biotype_df[biotype_df["transcript_id"].isin(adata_tx)].copy()
    
    if df.empty:
        print("No matching transcripts found.")
        return None
    
    # Count biotypes
    counts = df["biotype"].value_counts()
    total = counts.sum()
    
    # Group small categories into "other"
    top_biotypes = counts.head(top_n)
    if len(counts) > top_n:
        other_count = counts.iloc[top_n:].sum()
        top_biotypes["other"] = other_count
    
    # Calculate percentages
    percentages = top_biotypes / total * 100
    
    # Sort by count (descending)
    top_biotypes = top_biotypes.sort_values(ascending=True)
    percentages = percentages.reindex(top_biotypes.index)
    
    if colors is None:
        colors = ALLOS_PALETTE[:len(top_biotypes)]
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    # Horizontal bar chart
    bars = ax.barh(range(len(top_biotypes)), top_biotypes.values, 
                   color=colors[::-1], edgecolor='white', linewidth=1.5)
    
    ax.set_yticks(range(len(top_biotypes)))
    ax.set_yticklabels(top_biotypes.index)
    ax.set_xlabel("Number of transcripts")
    ax.set_title(title, pad=15)
    ax.xaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Add count and percentage labels
    for i, (count, pct) in enumerate(zip(top_biotypes.values, percentages.values)):
        ax.text(count + total * 0.01, i, f'{count:,} ({pct:.1f}%)',
               va='center', fontsize=9, color='#333333')
    
    # Extend x-axis to fit labels
    ax.set_xlim(0, top_biotypes.max() * 1.25)
    
    # Add total annotation
    ax.text(0.98, 0.02, f'Total: {total:,} transcripts',
           transform=ax.transAxes, ha='right', va='bottom',
           fontsize=10, color='#666666')
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return pd.DataFrame({"biotype": top_biotypes.index, 
                            "count": top_biotypes.values,
                            "percent": percentages.values}), fig
    return None


def plot_biotype_by_group(
    td,
    adata,
    groupby: str,
    *,
    adata_tx_col: Optional[str] = None,
    layer: Optional[str] = None,
    min_total: int = 1,
    top_n_biotypes: int = 6,
    normalize: bool = True,
    title: Optional[str] = None,
    palette: Optional[List] = None,
    figsize: Optional[Tuple[float, float]] = None,
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Stacked bar chart showing biotype composition across groups (e.g., cell types).
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object.
    adata : AnnData
        AnnData object with transcript-level data.
    groupby : str
        Column in adata.obs to group by.
    normalize : bool
        If True, show percentages (stacked to 100%). If False, show counts.
    top_n_biotypes : int
        Number of biotypes to show (rest grouped as "other").
    """
    _setup_scanpy_style()
    
    # Get biotype info
    biotype_dict = {}
    for tid, rec in td._idx._tx_by_id.items():
        tid_stripped = tid.split(".")[0]
        biotype = rec.transcript_type if rec.transcript_type and rec.transcript_type != "nan" else "unknown"
        biotype_dict[tid_stripped] = biotype
    
    if adata_tx_col is None:
        adata_tx = list(_strip_ver(adata.var.index))
    else:
        adata_tx = list(_strip_ver(adata.var[adata_tx_col].values))
    
    X, _ = _get_matrix(adata, layer)
    groups = adata.obs[groupby].unique()
    
    # Get top biotypes globally
    all_biotypes = [biotype_dict.get(tx, "unknown") for tx in adata_tx]
    global_counts = pd.Series(all_biotypes).value_counts()
    top_biotypes = global_counts.head(top_n_biotypes).index.tolist()
    
    # Compute biotype counts per group
    results = []
    for g in groups:
        mask = (adata.obs[groupby].values == g)
        Xg = X[mask]
        
        # Sum expression per transcript
        expr_sum = np.asarray(Xg.sum(axis=0)).ravel() if sparse.issparse(Xg) else Xg.sum(axis=0)
        
        # Count expressed transcripts by biotype
        biotype_counts = {}
        for i, tx in enumerate(adata_tx):
            if expr_sum[i] >= min_total:
                bt = biotype_dict.get(tx, "unknown")
                bt_plot = bt if bt in top_biotypes else "other"
                biotype_counts[bt_plot] = biotype_counts.get(bt_plot, 0) + 1
        
        for bt, count in biotype_counts.items():
            results.append({"group": g, "biotype": bt, "count": count})
    
    df = pd.DataFrame(results)
    
    if df.empty:
        print("No data found.")
        return None
    
    # Pivot for stacked bar
    pivot = df.pivot(index="group", columns="biotype", values="count").fillna(0)
    
    # Ensure consistent column order
    col_order = [bt for bt in top_biotypes if bt in pivot.columns]
    if "other" in pivot.columns:
        col_order.append("other")
    pivot = pivot[col_order]
    
    if normalize:
        pivot = pivot.div(pivot.sum(axis=1), axis=0) * 100
        ylabel = "Percentage of transcripts"
    else:
        ylabel = "Number of transcripts"
    
    if figsize is None:
        figsize = (max(7, len(groups) * 0.8 + 2), 5)
    
    if palette is None:
        palette = ALLOS_PALETTE[:len(col_order)]
    
    if title is None:
        title = f"Biotype Composition by {groupby}"
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    pivot.plot(kind='bar', stacked=True, ax=ax, color=palette, 
               edgecolor='white', linewidth=0.5, width=0.8)
    
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=15)
    ax.legend(title="Biotype", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.xticks(rotation=45, ha='right')
    
    if normalize:
        ax.set_ylim(0, 100)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0f}%'))
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return pivot, fig
    return None


def plot_length_expression_correlation(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    layer: Optional[str] = None,
    color_by: str = "density",
    log_expression: bool = True,
    log_length: bool = True,  # Changed default to True
    min_expression: float = 0,  # New: filter low expression
    show_marginals: bool = True,  # New: marginal histograms
    title: str = "Transcript Length vs Expression",
    figsize: Tuple[float, float] = (8, 8),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Scatter plot of transcript length vs mean expression with marginal histograms.
    
    Useful for identifying length biases in detection.
    """
    import seaborn as sns
    from scipy.stats import pearsonr, spearmanr
    _setup_scanpy_style()
    
    length_table = td.transcript_lengths_table()
    
    if adata_tx_col is None:
        adata_tx = list(_strip_ver(adata.var.index))
    else:
        adata_tx = list(_strip_ver(adata.var[adata_tx_col].values))
    
    length_dict = dict(zip(length_table["transcript_id"], length_table["length"]))
    gene_dict = dict(zip(length_table["transcript_id"], length_table["gene"]))
    
    X, _ = _get_matrix(adata, layer)
    mean_expr = np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else X.mean(axis=0)
    
    df = pd.DataFrame({
        "transcript_id": adata_tx,
        "length": [length_dict.get(tx, 0) for tx in adata_tx],
        "mean_expression": mean_expr,
        "gene": [gene_dict.get(tx, "") for tx in adata_tx]
    })
    
    # Filter
    df = df[(df["length"] > 0) & (df["mean_expression"] >= min_expression)].copy()
    
    if df.empty:
        print("No transcripts with valid lengths found.")
        return None
    
    # Transform
    df["expr_plot"] = np.log1p(df["mean_expression"]) if log_expression else df["mean_expression"]
    df["length_plot"] = np.log10(df["length"]) if log_length else df["length"]
    
    expr_label = "Mean expression (log(x+1))" if log_expression else "Mean expression"
    length_label = "Transcript length (log₁₀ bp)" if log_length else "Transcript length (bp)"
    
    # Calculate correlations
    r_pearson, _ = pearsonr(df["length_plot"], df["expr_plot"])
    r_spearman, _ = spearmanr(df["length_plot"], df["expr_plot"])
    
    if show_marginals and color_by == "density":
        # Use seaborn jointplot for marginals
        g = sns.JointGrid(data=df, x="length_plot", y="expr_plot", height=figsize[0]*0.9)
        
        g.plot_joint(sns.histplot, bins=60, pthresh=0.05, cmap="Blues", cbar=True)
        g.plot_marginals(sns.histplot, kde=True, color=ALLOS_PALETTE[0], alpha=0.7)
        
        # Add trend line
        z = np.polyfit(df["length_plot"], df["expr_plot"], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df["length_plot"].min(), df["length_plot"].max(), 100)
        g.ax_joint.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)
        
        # Add correlation text
        g.ax_joint.text(0.02, 0.98, f"Pearson r = {r_pearson:.3f}\nSpearman ρ = {r_spearman:.3f}",
                       transform=g.ax_joint.transAxes, ha='left', va='top', fontsize=10,
                       bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='none'))
        
        g.ax_joint.set_xlabel(length_label)
        g.ax_joint.set_ylabel(expr_label)
        g.fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
        
        # Add reference lines for kb markers
        if log_length:
            for kb in [1, 5, 10]:
                if np.log10(kb * 1000) < g.ax_joint.get_xlim()[1]:
                    g.ax_joint.axvline(np.log10(kb * 1000), color='gray', linestyle=':', alpha=0.4)
        
        g.ax_joint.text(0.98, 0.02, f"n = {len(df):,} transcripts", transform=g.ax_joint.transAxes,
                       ha='right', va='bottom', fontsize=9, color='#666666')
        
        fig = g.fig
        
    else:
        fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
        
        if color_by == "density":
            hb = ax.hexbin(df["length_plot"], df["expr_plot"], gridsize=50, cmap="Blues", mincnt=1)
            plt.colorbar(hb, ax=ax, label="Count")
        elif color_by == "gene":
            gene_counts = df["gene"].value_counts()
            df["n_isoforms"] = df["gene"].map(gene_counts)
            scatter = ax.scatter(df["length_plot"], df["expr_plot"], c=df["n_isoforms"],
                               cmap="viridis", alpha=0.6, s=15, edgecolors='none')
            plt.colorbar(scatter, ax=ax, label="Isoforms per gene")
        else:
            ax.scatter(df["length_plot"], df["expr_plot"], alpha=0.5, s=10,
                      color=ALLOS_PALETTE[0], edgecolors='none')
        
        # Trend line
        z = np.polyfit(df["length_plot"], df["expr_plot"], 1)
        p = np.poly1d(z)
        x_line = np.linspace(df["length_plot"].min(), df["length_plot"].max(), 100)
        ax.plot(x_line, p(x_line), "r--", alpha=0.8, linewidth=2)
        
        ax.text(0.02, 0.98, f"Pearson r = {r_pearson:.3f}\nSpearman ρ = {r_spearman:.3f}",
               transform=ax.transAxes, ha='left', va='top', fontsize=10,
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='none'))
        
        ax.set_xlabel(length_label)
        ax.set_ylabel(expr_label)
        ax.set_title(title, pad=15)
        ax.yaxis.grid(True, linestyle='--', alpha=0.3)
        ax.xaxis.grid(True, linestyle='--', alpha=0.3)
        ax.set_axisbelow(True)
        
        ax.text(0.98, 0.02, f"n = {len(df):,}", transform=ax.transAxes,
               ha='right', va='bottom', fontsize=9, color='#666666')
        
        plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return df, fig
    return None


def plot_length_bias(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    n_bins: int = 25,
    title: str = "Length Bias Assessment",
    figsize: Tuple[float, float] = (12, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Compare length distribution of ALL transcripts in GTF vs those DETECTED in your data.
    
    This reveals systematic length bias - if longer transcripts are under-represented
    in your detected set compared to the annotation, you have length bias.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object (contains all GTF transcripts).
    adata : AnnData
        AnnData object with detected transcripts.
    adata_tx_col : str, optional
        Column in adata.var containing transcript IDs.
    n_bins : int
        Number of bins for the histogram.
    
    Returns
    -------
    Shows three panels:
        1. Overlaid histograms (GTF vs Detected)
        2. Enrichment ratio (Detected/Expected) by length bin
        3. Cumulative distribution comparison
    """
    import seaborn as sns
    from scipy import stats
    _setup_scanpy_style()
    
    # Get ALL transcripts from GTF with their lengths
    gtf_lengths = []
    for tid, rec in td._idx._tx_by_id.items():
        length = int(np.sum(rec.exons[:, 1] - rec.exons[:, 0])) if rec.exons.size > 0 else 0
        if length > 0:
            gtf_lengths.append(length)
    gtf_lengths = np.array(gtf_lengths)
    
    # Get DETECTED transcripts from adata
    if adata_tx_col is None:
        adata_tx = set(_strip_ver(adata.var.index))
    else:
        adata_tx = set(_strip_ver(adata.var[adata_tx_col].values))
    
    detected_lengths = []
    for tid, rec in td._idx._tx_by_id.items():
        tid_stripped = tid.split(".")[0]
        if tid_stripped in adata_tx:
            length = int(np.sum(rec.exons[:, 1] - rec.exons[:, 0])) if rec.exons.size > 0 else 0
            if length > 0:
                detected_lengths.append(length)
    detected_lengths = np.array(detected_lengths)
    
    if len(detected_lengths) == 0:
        print("No matching transcripts found.")
        return None
    
    # Log transform for visualization
    gtf_log = np.log10(gtf_lengths)
    detected_log = np.log10(detected_lengths)
    
    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=dpi)
    
    # Panel 1: Overlaid histograms (normalized)
    bins = np.linspace(min(gtf_log.min(), detected_log.min()), 
                       max(gtf_log.max(), detected_log.max()), n_bins)
    
    axes[0].hist(gtf_log, bins=bins, density=True, alpha=0.5, 
                 color=ALLOS_PALETTE[0], label=f'GTF annotation (n={len(gtf_lengths):,})', 
                 edgecolor='white', linewidth=0.5)
    axes[0].hist(detected_log, bins=bins, density=True, alpha=0.5,
                 color=ALLOS_PALETTE[3], label=f'Detected (n={len(detected_lengths):,})',
                 edgecolor='white', linewidth=0.5)
    
    axes[0].set_xlabel("Transcript length (log₁₀ bp)")
    axes[0].set_ylabel("Density")
    axes[0].set_title("Length Distribution Comparison")
    axes[0].legend(fontsize=8)
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].set_axisbelow(True)
    
    # Add kb markers
    for kb in [1, 5, 10]:
        axes[0].axvline(np.log10(kb * 1000), color='gray', linestyle=':', alpha=0.4)
    
    # Panel 2: Enrichment ratio by length bin
    gtf_counts, bin_edges = np.histogram(gtf_log, bins=bins)
    detected_counts, _ = np.histogram(detected_log, bins=bin_edges)
    
    # Normalize to get expected proportions
    gtf_prop = gtf_counts / gtf_counts.sum()
    detected_prop = detected_counts / detected_counts.sum()
    
    # Enrichment ratio (detected/expected), avoiding division by zero
    with np.errstate(divide='ignore', invalid='ignore'):
        enrichment = np.where(gtf_prop > 0, detected_prop / gtf_prop, np.nan)
    
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    
    # Color bars by enrichment (red = depleted, blue = enriched)
    colors = ['#e74c3c' if e < 0.8 else '#27ae60' if e > 1.2 else '#95a5a6' 
              for e in enrichment]
    
    bars = axes[1].bar(bin_centers, enrichment, width=np.diff(bin_edges)[0] * 0.9,
                       color=colors, edgecolor='white', linewidth=0.5)
    
    axes[1].axhline(1.0, color='black', linestyle='-', linewidth=1.5, label='No bias')
    axes[1].axhline(0.8, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.7)
    axes[1].axhline(1.2, color='#27ae60', linestyle='--', linewidth=1, alpha=0.7)
    
    axes[1].set_xlabel("Transcript length (log₁₀ bp)")
    axes[1].set_ylabel("Enrichment ratio (Detected / Expected)")
    axes[1].set_title("Length Bias (ratio)")
    axes[1].set_ylim(0, min(2.5, np.nanmax(enrichment) * 1.1) if np.any(np.isfinite(enrichment)) else 2.5)
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[1].set_axisbelow(True)
    
    # Add colorbar legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#e74c3c', label='Depleted (<0.8)'),
                       Patch(facecolor='#95a5a6', label='Normal (0.8-1.2)'),
                       Patch(facecolor='#27ae60', label='Enriched (>1.2)')]
    axes[1].legend(handles=legend_elements, fontsize=7, loc='upper right')
    
    # Panel 3: Cumulative distribution (more sensitive to bias)
    gtf_sorted = np.sort(gtf_log)
    detected_sorted = np.sort(detected_log)
    
    gtf_cdf = np.arange(1, len(gtf_sorted) + 1) / len(gtf_sorted)
    detected_cdf = np.arange(1, len(detected_sorted) + 1) / len(detected_sorted)
    
    axes[2].plot(gtf_sorted, gtf_cdf, color=ALLOS_PALETTE[0], linewidth=2, label='GTF (expected)')
    axes[2].plot(detected_sorted, detected_cdf, color=ALLOS_PALETTE[3], linewidth=2, label='Detected')
    
    # KS test
    ks_stat, ks_pval = stats.ks_2samp(gtf_log, detected_log)
    
    axes[2].set_xlabel("Transcript length (log₁₀ bp)")
    axes[2].set_ylabel("Cumulative proportion")
    axes[2].set_title("Cumulative Distribution")
    axes[2].legend(fontsize=8)
    axes[2].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[2].set_axisbelow(True)
    
    # Add KS test result
    axes[2].text(0.98, 0.02, f'KS test: D={ks_stat:.3f}, p={ks_pval:.2e}',
                transform=axes[2].transAxes, ha='right', va='bottom', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Overall bias assessment
    median_gtf = np.median(gtf_lengths)
    median_detected = np.median(detected_lengths)
    bias_ratio = median_detected / median_gtf
    
    if bias_ratio < 0.9:
        verdict = f"⚠️ Short transcript bias (median ratio: {bias_ratio:.2f})"
        verdict_color = '#e74c3c'
    elif bias_ratio > 1.1:
        verdict = f"⚠️ Long transcript bias (median ratio: {bias_ratio:.2f})"
        verdict_color = '#3498db'
    else:
        verdict = f"✓ No significant length bias (median ratio: {bias_ratio:.2f})"
        verdict_color = '#27ae60'
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    fig.text(0.5, -0.02, verdict, ha='center', fontsize=11, fontweight='bold', color=verdict_color)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "gtf_lengths": gtf_lengths,
            "detected_lengths": detected_lengths,
            "enrichment": enrichment,
            "bin_centers": bin_centers,
            "ks_stat": ks_stat,
            "ks_pval": ks_pval,
            "median_ratio": bias_ratio
        }, fig
    return None


def plot_length_bias_by_expression(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    layer: Optional[str] = None,
    n_expr_bins: int = 5,
    title: str = "Length Bias by Expression Level",
    figsize: Tuple[float, float] = (12, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    For transcripts at similar expression levels, are longer ones detected in fewer cells?
    
    This controls for expression level to reveal true length bias in detection.
    If there's length bias, within each expression bin, longer transcripts
    should have lower detection rates.
    
    Parameters
    ----------
    td : TranscriptData
        TranscriptData object.
    adata : AnnData
        AnnData object with transcript-level data.
    n_expr_bins : int
        Number of expression level bins.
    """
    import seaborn as sns
    from scipy import stats
    _setup_scanpy_style()
    
    # Get lengths
    length_table = td.transcript_lengths_table()
    
    if adata_tx_col is None:
        adata_tx = list(_strip_ver(adata.var.index))
    else:
        adata_tx = list(_strip_ver(adata.var[adata_tx_col].values))
    
    length_dict = dict(zip(length_table["transcript_id"], length_table["length"]))
    
    X, _ = _get_matrix(adata, layer)
    n_cells = X.shape[0]
    
    # Calculate metrics per transcript
    mean_expr = np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else X.mean(axis=0)
    if sparse.issparse(X):
        detection_rate = np.asarray((X > 0).sum(axis=0)).ravel() / n_cells * 100
    else:
        detection_rate = (X > 0).sum(axis=0) / n_cells * 100
    
    df = pd.DataFrame({
        "transcript_id": adata_tx,
        "length": [length_dict.get(tx, 0) for tx in adata_tx],
        "mean_expression": mean_expr,
        "detection_rate": detection_rate
    })
    
    # Filter valid
    df = df[(df["length"] > 0) & (df["mean_expression"] > 0)].copy()
    
    if df.empty:
        print("No valid transcripts found.")
        return None
    
    # Log transform
    df["log_length"] = np.log10(df["length"])
    df["log_expr"] = np.log1p(df["mean_expression"])
    
    # Bin by expression level
    df["expr_bin"] = pd.qcut(df["log_expr"], n_expr_bins, labels=False, duplicates='drop')
    expr_bin_labels = df.groupby("expr_bin")["mean_expression"].apply(
        lambda x: f"{x.min():.2f}-{x.max():.2f}"
    ).to_dict()
    df["expr_bin_label"] = df["expr_bin"].map(expr_bin_labels)
    
    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=dpi)
    
    # Panel 1: Scatter colored by expression bin
    scatter = axes[0].scatter(df["log_length"], df["detection_rate"], 
                              c=df["expr_bin"], cmap="viridis", 
                              alpha=0.5, s=10, edgecolors='none')
    
    axes[0].set_xlabel("Transcript length (log₁₀ bp)")
    axes[0].set_ylabel("Detection rate (% cells)")
    axes[0].set_title("Detection vs Length\n(colored by expression)")
    cbar = plt.colorbar(scatter, ax=axes[0])
    cbar.set_label("Expression bin", fontsize=8)
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].set_axisbelow(True)
    
    # Panel 2: Within each expression bin, correlation of length vs detection
    correlations = []
    for expr_bin in sorted(df["expr_bin"].unique()):
        bin_data = df[df["expr_bin"] == expr_bin]
        if len(bin_data) > 10:
            r, p = stats.spearmanr(bin_data["log_length"], bin_data["detection_rate"])
            correlations.append({
                "expr_bin": expr_bin,
                "correlation": r,
                "p_value": p,
                "n": len(bin_data),
                "label": expr_bin_labels.get(expr_bin, str(expr_bin))
            })
    
    corr_df = pd.DataFrame(correlations)
    
    colors = ['#e74c3c' if c < -0.1 else '#27ae60' if c > 0.1 else '#95a5a6' 
              for c in corr_df["correlation"]]
    
    bars = axes[1].bar(range(len(corr_df)), corr_df["correlation"], color=colors,
                       edgecolor='white', linewidth=1)
    
    axes[1].axhline(0, color='black', linewidth=1)
    axes[1].axhline(-0.1, color='#e74c3c', linestyle='--', alpha=0.5)
    axes[1].axhline(0.1, color='#27ae60', linestyle='--', alpha=0.5)
    
    axes[1].set_xticks(range(len(corr_df)))
    axes[1].set_xticklabels([f"Bin {i+1}" for i in range(len(corr_df))], fontsize=8)
    axes[1].set_xlabel("Expression level bin (low → high)")
    axes[1].set_ylabel("Spearman correlation\n(length vs detection)")
    axes[1].set_title("Length-Detection Correlation\nby Expression Level")
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[1].set_axisbelow(True)
    
    # Add significance markers
    for i, row in corr_df.iterrows():
        sig = "***" if row["p_value"] < 0.001 else "**" if row["p_value"] < 0.01 else "*" if row["p_value"] < 0.05 else ""
        y_pos = row["correlation"] + 0.02 if row["correlation"] >= 0 else row["correlation"] - 0.05
        axes[1].text(i, y_pos, sig, ha='center', fontsize=10, fontweight='bold')
    
    # Panel 3: Boxplot of length by expression bin
    sns.boxplot(data=df, x="expr_bin", y="log_length", ax=axes[2],
               palette="viridis", linewidth=1)
    
    axes[2].set_xlabel("Expression level bin (low → high)")
    axes[2].set_ylabel("Transcript length (log₁₀ bp)")
    axes[2].set_title("Length Distribution\nby Expression Level")
    axes[2].set_xticklabels([f"Bin {i+1}" for i in range(len(df["expr_bin"].unique()))], fontsize=8)
    axes[2].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[2].set_axisbelow(True)
    
    # Overall verdict
    mean_corr = corr_df["correlation"].mean()
    if mean_corr < -0.15:
        verdict = f"⚠️ Length bias detected: longer transcripts less detected (mean r = {mean_corr:.2f})"
        verdict_color = '#e74c3c'
    elif mean_corr > 0.15:
        verdict = f"ℹ️ Longer transcripts more detected (mean r = {mean_corr:.2f})"
        verdict_color = '#3498db'
    else:
        verdict = f"✓ No significant length bias after expression normalization (mean r = {mean_corr:.2f})"
        verdict_color = '#27ae60'
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    fig.text(0.5, -0.02, verdict, ha='center', fontsize=10, fontweight='bold', color=verdict_color)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {"transcript_data": df, "correlations": corr_df}, fig
    return None


def plot_end_bias_note():
    """
    Note: 5'/3' end bias analysis requires read-level data (BAM files).
    
    To assess whether transcripts are truncated at the 5' or 3' end,
    you need to analyze the coverage profile across transcript bodies
    from aligned reads.
    
    Recommended tools for this analysis:
    - RSeQC (geneBody_coverage.py)
    - Picard CollectRnaSeqMetrics
    - Custom analysis with pysam
    
    What to look for:
    - Flat coverage = good full-length capture
    - 3' bias (common in poly-A selection) = higher coverage at 3' end
    - 5' dropout (cDNA synthesis issue) = lower coverage at 5' end
    
    This function is a placeholder - implement with BAM data if available.
    """
    print("="*60)
    print("5'/3' END BIAS ANALYSIS")
    print("="*60)
    print()
    print("This analysis requires read-level data (BAM files).")
    print()
    print("To assess end bias, you need to analyze coverage profiles")
    print("across transcript bodies from aligned reads.")
    print()
    print("Recommended approaches:")
    print("  1. RSeQC geneBody_coverage.py")
    print("  2. Picard CollectRnaSeqMetrics")
    print("  3. Custom analysis with pysam")
    print()
    print("What to look for:")
    print("  • Flat coverage = good full-length capture")
    print("  • 3' bias = higher coverage at 3' end (poly-A selection)")
    print("  • 5' dropout = lower coverage at 5' end (RT issues)")
    print()
    print("If you have BAM files, provide the path and I can help")
    print("implement a proper end bias analysis.")
    print("="*60)




def plot_gene_body_coverage(
    bam_path: str,
    td,
    *,
    n_bins: int = 100,
    max_transcripts: int = 2000,
    min_reads: int = 10,
    min_length: int = 500,
    stranded: bool = True,
    title: str = "Gene Body Coverage (5' → 3')",
    figsize: Tuple[float, float] = (10, 6),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot gene body coverage profile from 5' to 3' end using BAM file.
    
    OPTIMIZED: Uses count_coverage for fast pre-filtering before detailed analysis.
    
    Works with GENOME-aligned BAMs by mapping genomic coordinates to transcript positions.
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file (genome-aligned).
    td : TranscriptData
        TranscriptData object with transcript annotations.
    n_bins : int
        Number of bins along transcript body (default 100).
    max_transcripts : int
        Maximum transcripts to sample for speed.
    min_reads : int
        Minimum reads per transcript to include.
    min_length : int
        Minimum transcript length (bp) to include.
    stranded : bool
        If True, only count reads matching transcript strand (default True).
        For single-cell RNA-seq with poly-A priming, should be True.
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM file: {e}")
        return None
    
    # ========== STEP 1: Fast pre-filtering with count_coverage ==========
    print("Step 1: Fast pre-filtering transcripts with count_coverage...")
    
    candidate_transcripts = []  # (tid, total_depth, tx_length, strand, exons, chrom)
    n_checked = 0
    n_too_short = 0
    
    for tid, rec in td._idx._tx_by_id.items():
        chrom = rec.chrom
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        if tx_length < min_length:
            n_too_short += 1
            continue
        
        # Fast coverage check using count_coverage (C-level, much faster)
        total_depth = 0
        try:
            for exon_start, exon_end in exons:
                exon_start, exon_end = int(exon_start), int(exon_end)
                cov_tuple = bam.count_coverage(chrom, exon_start, exon_end,
                                               quality_threshold=0,
                                               read_callback='nofilter')
                for arr in cov_tuple:
                    total_depth += sum(arr)
        except:
            continue
        
        # Only keep transcripts with sufficient coverage
        if total_depth >= min_reads:
            candidate_transcripts.append((tid, total_depth, tx_length, rec.strand, exons, chrom))
        
        n_checked += 1
        if n_checked % 5000 == 0:
            print(f"  Checked {n_checked} transcripts, {len(candidate_transcripts)} pass filter...")
    
    print(f"  Found {len(candidate_transcripts)} transcripts with >= {min_reads} reads")
    print(f"  Skipped {n_too_short} transcripts shorter than {min_length}bp")
    
    # Randomly sample from candidates
    np.random.shuffle(candidate_transcripts)
    candidates_to_process = candidate_transcripts[:max_transcripts]
    
    # ========== STEP 2: Detailed coverage analysis ==========
    print(f"Step 2: Computing detailed coverage for {len(candidates_to_process)} transcripts...")
    
    coverage_profiles = []
    n_processed = 0
    n_skipped_no_reads = 0
    
    for tid, total_depth, tx_length, strand, exons, chrom in candidates_to_process:
        exons_sorted = exons[np.argsort(exons[:, 0])]
        tx_coverage = np.zeros(tx_length)
        
        try:
            tx_pos = 0
            for exon_start, exon_end in exons_sorted:
                exon_start, exon_end = int(exon_start), int(exon_end)
                exon_len = exon_end - exon_start
                
                for read in bam.fetch(chrom, exon_start, exon_end):
                    if read.is_unmapped or read.is_secondary or read.is_supplementary:
                        continue
                    
                    if stranded:
                        read_on_plus = not read.is_reverse
                        tx_on_plus = (strand == 1)
                        if read_on_plus != tx_on_plus:
                            continue
                    
                    read_start = max(read.reference_start, exon_start)
                    read_end = min(read.reference_end or read.reference_start + 100, exon_end)
                    
                    if read_end <= read_start:
                        continue
                    
                    tx_start = tx_pos + (read_start - exon_start)
                    tx_end = tx_pos + (read_end - exon_start)
                    tx_start = max(0, min(tx_start, tx_length - 1))
                    tx_end = max(0, min(tx_end, tx_length))
                    
                    tx_coverage[int(tx_start):int(tx_end)] += 1
                
                tx_pos += exon_len
            
            total_reads = tx_coverage.sum()
            if total_reads < min_reads:
                n_skipped_no_reads += 1
                continue
            
            # Normalize coverage
            if tx_coverage.max() > 0:
                tx_coverage = tx_coverage / tx_coverage.max()
            
            # Bin into n_bins
            bin_size = tx_length / n_bins
            binned = np.array([
                tx_coverage[int(i * bin_size):int((i + 1) * bin_size)].mean()
                for i in range(n_bins)
            ])
            
            # If minus strand, flip so 5'->3' is always left->right
            if strand == -1:
                binned = binned[::-1]
            
            coverage_profiles.append(binned)
            n_processed += 1
            
            if n_processed % 500 == 0:
                print(f"  Processed {n_processed} transcripts...")
            
        except Exception as e:
            continue
    
    bam.close()
    
    print(f"Step 3: Completed. Analyzed {n_processed} transcripts.")
    
    if len(coverage_profiles) == 0:
        print("No transcripts with sufficient coverage found.")
        return None
    
    # Aggregate
    mean_profile = np.mean(coverage_profiles, axis=0)
    std_profile = np.std(coverage_profiles, axis=0)
    
    # Plotting
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    x = np.linspace(0, 100, n_bins)
    
    ax.fill_between(x, mean_profile - std_profile, mean_profile + std_profile, 
                   alpha=0.2, color=ALLOS_PALETTE[0])
    ax.plot(x, mean_profile, color=ALLOS_PALETTE[0], linewidth=2, label='Mean coverage')
    
    ax.axhline(mean_profile.mean(), color='gray', linestyle='--', alpha=0.5, label='Overall mean')
    
    ax.set_xlabel("Normalized transcript position (%)")
    ax.set_ylabel("Normalized coverage")
    ax.set_title(f"{title}\n(n={n_processed:,} transcripts)")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=9)
    
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    ax.text(2, 1.02, "5'", fontsize=12, fontweight='bold')
    ax.text(96, 1.02, "3'", fontsize=12, fontweight='bold')
    
    # Calculate and display 5'/3' ratio
    five_prime = mean_profile[:20].mean()
    three_prime = mean_profile[-20:].mean()
    ratio = five_prime / three_prime if three_prime > 0 else 0
    
    ax.text(0.98, 0.02, f"5'/3' ratio: {ratio:.2f}", transform=ax.transAxes,
           ha='right', fontsize=10, 
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "mean_profile": mean_profile,
            "std_profile": std_profile,
            "n_transcripts": n_processed,
            "five_prime_ratio": ratio
        }, fig
    return None



def plot_gene_body_coverage_comparison(
    bam_paths: dict,
    td,
    *,
    n_bins: int = 100,
    max_transcripts: int = 1500,
    min_reads: int = 10,
    min_length: int = 500,
    stranded: bool = True,
    title: str = "Gene Body Coverage Comparison",
    figsize: Tuple[float, float] = (10, 6),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Compare gene body coverage profiles across multiple samples.
    
    OPTIMIZED: Uses count_coverage for fast pre-filtering.
    
    Parameters
    ----------
    bam_paths : dict
        Dictionary mapping sample names to BAM file paths.
    stranded : bool
        If True, only count reads matching transcript strand (default True).
    """
    import pysam
    _setup_scanpy_style()
    
    sample_profiles = {}
    sample_stats = {}
    
    for sample_name, bam_path in bam_paths.items():
        print(f"\nProcessing {sample_name}...")
        
        try:
            bam = pysam.AlignmentFile(bam_path, "rb")
        except Exception as e:
            print(f"  Error opening BAM: {e}")
            continue
        
        # ========== Fast pre-filtering with count_coverage ==========
        print(f"  Step 1: Fast pre-filtering...")
        
        candidate_transcripts = []
        n_checked = 0
        
        for tid, rec in td._idx._tx_by_id.items():
            chrom = rec.chrom
            exons = rec.exons
            
            if exons.size == 0:
                continue
            
            tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
            if tx_length < min_length:
                continue
            
            # Fast coverage check
            total_depth = 0
            try:
                for exon_start, exon_end in exons:
                    exon_start, exon_end = int(exon_start), int(exon_end)
                    cov_tuple = bam.count_coverage(chrom, exon_start, exon_end,
                                                   quality_threshold=0,
                                                   read_callback='nofilter')
                    for arr in cov_tuple:
                        total_depth += sum(arr)
            except:
                continue
            
            if total_depth >= min_reads:
                candidate_transcripts.append((tid, total_depth, tx_length, rec.strand, exons, chrom))
            
            n_checked += 1
            if n_checked % 10000 == 0:
                print(f"    Checked {n_checked} transcripts...")
        
        print(f"  Found {len(candidate_transcripts)} candidate transcripts")
        
        # Sample and process
        np.random.shuffle(candidate_transcripts)
        
        coverage_profiles = []
        n_processed = 0
        
        for tid, total_depth, tx_length, strand, exons, chrom in candidate_transcripts[:max_transcripts]:
            exons_sorted = exons[np.argsort(exons[:, 0])]
            tx_coverage = np.zeros(tx_length)
            
            try:
                tx_pos = 0
                for exon_start, exon_end in exons_sorted:
                    exon_start, exon_end = int(exon_start), int(exon_end)
                    exon_len = exon_end - exon_start
                    
                    for read in bam.fetch(chrom, exon_start, exon_end):
                        if read.is_unmapped or read.is_secondary or read.is_supplementary:
                            continue
                        
                        if stranded:
                            read_on_plus = not read.is_reverse
                            tx_on_plus = (strand == 1)
                            if read_on_plus != tx_on_plus:
                                continue
                        
                        read_start = max(read.reference_start, exon_start)
                        read_end = min(read.reference_end or read.reference_start + 100, exon_end)
                        
                        if read_end <= read_start:
                            continue
                        
                        tx_start = tx_pos + (read_start - exon_start)
                        tx_end = tx_pos + (read_end - exon_start)
                        tx_start = max(0, min(tx_start, tx_length - 1))
                        tx_end = max(0, min(tx_end, tx_length))
                        
                        tx_coverage[int(tx_start):int(tx_end)] += 1
                    
                    tx_pos += exon_len
                
                if tx_coverage.sum() < min_reads:
                    continue
                
                if tx_coverage.max() > 0:
                    tx_coverage = tx_coverage / tx_coverage.max()
                
                bin_size = tx_length / n_bins
                binned = np.array([
                    tx_coverage[int(i * bin_size):int((i + 1) * bin_size)].mean()
                    for i in range(n_bins)
                ])
                
                if strand == -1:
                    binned = binned[::-1]
                
                coverage_profiles.append(binned)
                n_processed += 1
                
            except:
                continue
        
        bam.close()
        
        if coverage_profiles:
            mean_profile = np.mean(coverage_profiles, axis=0)
            sample_profiles[sample_name] = mean_profile
            
            five_prime = mean_profile[:20].mean()
            three_prime = mean_profile[-20:].mean()
            
            sample_stats[sample_name] = {
                'n_transcripts': n_processed,
                'five_prime': five_prime,
                'three_prime': three_prime,
                'ratio': five_prime / three_prime if three_prime > 0 else 0
            }
            print(f"  Completed: {n_processed} transcripts, 5'/3' ratio: {sample_stats[sample_name]['ratio']:.2f}")
    
    if not sample_profiles:
        print("No samples with coverage found.")
        return None
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=dpi)
    
    x = np.linspace(0, 100, n_bins)
    colors = ALLOS_PALETTE[:len(sample_profiles)]
    
    # Panel 1: Coverage profiles
    for i, (name, profile) in enumerate(sample_profiles.items()):
        axes[0].plot(x, profile, linewidth=2, color=colors[i % len(colors)], 
                    label=f"{name} (n={sample_stats[name]['n_transcripts']})")
    
    axes[0].set_xlabel("Normalized position (%)")
    axes[0].set_ylabel("Normalized coverage")
    axes[0].set_title("Gene Body Coverage")
    axes[0].set_xlim(0, 100)
    axes[0].legend(fontsize=8)
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].text(2, axes[0].get_ylim()[1]*0.95, "5'", fontsize=11, fontweight='bold')
    axes[0].text(96, axes[0].get_ylim()[1]*0.95, "3'", fontsize=11, fontweight='bold')
    
    # Panel 2: 5'/3' ratios
    names = list(sample_stats.keys())
    ratios = [sample_stats[n]['ratio'] for n in names]
    bar_colors = ['#27ae60' if r > 0.8 else '#f39c12' if r > 0.5 else '#e74c3c' for r in ratios]
    
    bars = axes[1].bar(names, ratios, color=bar_colors, edgecolor='white', linewidth=1.5)
    axes[1].axhline(1.0, color='black', linestyle='-', linewidth=1.5)
    axes[1].axhline(0.8, color='gray', linestyle='--', alpha=0.5)
    axes[1].set_ylabel("5'/3' Ratio")
    axes[1].set_title("Coverage Bias")
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[1].tick_params(axis='x', rotation=45)
    
    for bar, ratio in zip(bars, ratios):
        axes[1].text(bar.get_x() + bar.get_width()/2, ratio + 0.02, f'{ratio:.2f}',
                    ha='center', fontsize=9, fontweight='bold')
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {"profiles": sample_profiles, "stats": sample_stats}, fig
    return None



def plot_gene_coverage(
    bam_path: str,
    td,
    *,
    adata=None,
    n_threads: int = 4,
    n_percentiles: int = 101,
    min_length: int = 500,
    end_bias_bases: int = 100,
    top_n_transcripts: int = 1000,
    max_depth: int = 5_000_000,
    quick_mode: bool = True,
    normalize_by: str = "max",
    title: str = "Gene Body Coverage",
    figsize: Tuple[float, float] = (12, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Gene body coverage replicating Picard CollectRnaSeqMetrics algorithm.
    
    FAST VERSION with multiple optimization strategies:
    - Option 1: Use adata expression to pre-select top transcripts (fastest)
    - Option 2: Skip hotspots with depth > max_depth
    - Option 3: Quick mode - sample first 100bp for ranking (default)
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file (must be coordinate-sorted and indexed).
    td : TranscriptData
        TranscriptData object with transcript annotations.
    adata : AnnData, optional
        If provided, uses adata.var[adata_rank_col] to select top transcripts.
        This is the FASTEST option - skips BAM scanning for ranking.
    
    n_threads : int
        Threads for BAM decompression (default 4).
    max_depth : int
        Skip transcripts with depth > this value (default 5M). Avoids hotspots.
    quick_mode : bool
        If True and adata not provided, only sample first 100bp for ranking.
    top_n_transcripts : int
        Number of top transcripts to analyze (default 1000).
    """
    import pysam
    import heapq
    
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb", threads=n_threads)
    except Exception as e:
        print(f"Error opening BAM file: {e}")
        return None
    
    print("=" * 60)
    print("GENE BODY COVERAGE ANALYSIS")
    print("=" * 60)
    
    # ========== STRATEGY SELECTION ==========
    if adata is not None:
        print("Strategy: Using adata expression (summed across cells) for transcript selection (FASTEST)")
        strategy = "adata"
    elif quick_mode:
        print("Strategy: Quick mode - sampling first 100bp for ranking")
        strategy = "quick"
    else:
        print("Strategy: Full scan (slower)")
        strategy = "full"
    
    print(f"Max depth threshold: {max_depth:,} (skip hotspots)")
    print(f"Target transcripts: {top_n_transcripts}")
    print("-" * 60)
    
    # ========== GET TOP TRANSCRIPTS ==========
    
    if strategy == "adata":
        # FASTEST: Use adata expression values (sum across cells)
        print("Step 1: Selecting top transcripts from adata expression...")
        
        # Sum expression across all cells to get total counts per transcript
        if hasattr(adata.X, 'toarray'):
            total_expr = np.array(adata.X.sum(axis=0)).flatten()
        else:
            total_expr = np.array(adata.X.sum(axis=0)).flatten()
        
        # Create dataframe with transcript IDs and expression
        expr_df = pd.DataFrame({
            'tid': adata.var.index,
            'total_expr': total_expr
        }).set_index('tid')
        
        # Match transcript IDs between adata and td
        td_tids = set(td._idx._tx_by_id.keys())
        expr_df = expr_df[expr_df.index.isin(td_tids)]
        
        # Filter to min_length
        valid_tids = []
        for tid in expr_df.index:
            rec = td._idx._tx_by_id.get(tid)
            if rec is not None and rec.exons.size > 0:
                tx_len = int(np.sum(rec.exons[:, 1] - rec.exons[:, 0]))
                if tx_len >= min_length:
                    valid_tids.append(tid)
        expr_df = expr_df.loc[valid_tids]
        
        top_tids = expr_df.nlargest(top_n_transcripts, 'total_expr').index.tolist()
        print(f"  Found {len(expr_df)} transcripts in both adata and annotation")
        print(f"  Selected top {len(top_tids)} by expression")
        
    else:
        # QUICK or FULL: Scan BAM to rank transcripts
        print("Step 1: Scanning transcripts to estimate expression...")
        
        transcript_scores = []  # (score, tid)
        n_checked = 0
        n_skipped_short = 0
        n_skipped_hotspot = 0
        
        total = len(td._idx._tx_by_id)
        report_every = max(1, total // 20)
        
        for tid, rec in td._idx._tx_by_id.items():
            chrom = rec.chrom
            exons = rec.exons
            
            if exons.size == 0:
                continue
            
            tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
            if tx_length < min_length:
                n_skipped_short += 1
                continue
            
            # Get coverage estimate
            try:
                exon_start, exon_end = int(exons[0][0]), int(exons[0][1])
                
                if strategy == "quick":
                    # Only check first 100bp
                    check_end = min(exon_start + 100, exon_end)
                else:
                    check_end = exon_end
                
                cov_tuple = bam.count_coverage(chrom, exon_start, check_end,
                                               quality_threshold=0,
                                               read_callback='nofilter')
                depth = sum(sum(arr) for arr in cov_tuple)
                
                # Skip hotspots
                if depth > max_depth:
                    n_skipped_hotspot += 1
                    continue
                
                # Normalize by region length
                region_len = check_end - exon_start
                score = depth / region_len if region_len > 0 else 0
                
                transcript_scores.append((score, tid))
                
            except Exception as e:
                continue
            
            n_checked += 1
            if n_checked % report_every == 0:
                print(f"  Checked {n_checked}/{total} transcripts...")
        
        print(f"  Checked {n_checked} transcripts")
        print(f"  Skipped {n_skipped_short} (too short), {n_skipped_hotspot} (hotspots)")
        
        # Get top N
        transcript_scores.sort(reverse=True)
        top_tids = [tid for score, tid in transcript_scores[:top_n_transcripts]]
        print(f"  Selected top {len(top_tids)} transcripts")
    
    # ========== COMPUTE DETAILED COVERAGE ==========
    print("-" * 60)
    print(f"Step 2: Computing detailed coverage for {len(top_tids)} transcripts...")
    
    results = []
    n_processed = 0
    n_skipped_hotspot = 0
    report_every = max(1, len(top_tids) // 10)
    
    for tid in top_tids:
        rec = td._idx._tx_by_id.get(tid)
        if rec is None:
            continue
        
        chrom = rec.chrom
        strand = rec.strand
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        exons_sorted = exons[np.argsort(exons[:, 0])]
        
        # Get full coverage
        tx_coverage = np.zeros(tx_length, dtype=np.float32)
        tx_pos = 0
        total_depth = 0
        skip_transcript = False
        
        try:
            for exon_start, exon_end in exons_sorted:
                exon_start, exon_end = int(exon_start), int(exon_end)
                exon_len = exon_end - exon_start
                
                cov_tuple = bam.count_coverage(chrom, exon_start, exon_end,
                                               quality_threshold=0,
                                               read_callback='nofilter')
                
                exon_cov = np.zeros(exon_len, dtype=np.float32)
                for arr in cov_tuple:
                    exon_cov += np.array(arr, dtype=np.float32)
                
                exon_depth = exon_cov.sum()
                total_depth += exon_depth
                
                # Check for hotspot
                if total_depth > max_depth:
                    skip_transcript = True
                    break
                
                tx_coverage[tx_pos:tx_pos + exon_len] = exon_cov
                tx_pos += exon_len
            
            if skip_transcript:
                n_skipped_hotspot += 1
                continue
            
            if total_depth == 0:
                continue
            
            # Flip for minus strand
            if strand == -1:
                tx_coverage = tx_coverage[::-1]
            
            # Normalize
            mean_cov = tx_coverage.mean()
            max_cov = tx_coverage.max()
            if mean_cov == 0 or max_cov == 0:
                continue
            
            norm_factor = max_cov if normalize_by == "max" else mean_cov
            
            # Calculate bias
            five_prime_cov = tx_coverage[:end_bias_bases].mean() if tx_length >= end_bias_bases else tx_coverage.mean()
            three_prime_cov = tx_coverage[-end_bias_bases:].mean() if tx_length >= end_bias_bases else tx_coverage.mean()
            five_prime_bias = five_prime_cov / mean_cov
            three_prime_bias = three_prime_cov / mean_cov
            
            # Sample percentiles
            percentile_values = np.zeros(n_percentiles, dtype=np.float32)
            last_index = tx_length - 1
            for percent in range(n_percentiles):
                p = percent / 100.0
                start_idx = int(max(0, last_index * (p - 0.005)))
                end_idx = int(min(last_index, last_index * (p + 0.005)))
                if end_idx >= start_idx:
                    percentile_values[percent] = tx_coverage[start_idx:end_idx + 1].mean() / norm_factor
            
            results.append({
                'percentile_values': percentile_values,
                'five_prime_bias': five_prime_bias,
                'three_prime_bias': three_prime_bias
            })
            
            n_processed += 1
            if n_processed % report_every == 0:
                print(f"  Processed {n_processed}/{len(top_tids)} transcripts...")
                
        except Exception as e:
            continue
    
    bam.close()
    
    print(f"  Done! Processed {n_processed} transcripts, skipped {n_skipped_hotspot} hotspots")
    
    if n_processed == 0:
        print("ERROR: No transcripts with sufficient coverage found.")
        return None
    
    # ========== AGGREGATE RESULTS ==========
    print("-" * 60)
    print("Step 3: Aggregating results...")
    
    coverage_by_position = {p: [] for p in range(n_percentiles)}
    five_prime_biases = []
    three_prime_biases = []
    
    for result in results:
        for p, val in enumerate(result['percentile_values']):
            coverage_by_position[p].append(val)
        five_prime_biases.append(result['five_prime_bias'])
        three_prime_biases.append(result['three_prime_bias'])
    
    mean_coverage = np.array([
        np.mean(coverage_by_position[p]) if coverage_by_position[p] else 0
        for p in range(n_percentiles)
    ])
    
    median_5prime_bias = np.median(five_prime_biases)
    median_3prime_bias = np.median(three_prime_biases)
    median_5to3_ratio = median_5prime_bias / median_3prime_bias if median_3prime_bias > 0 else np.nan
    
    print(f"  5' bias: {median_5prime_bias:.3f}")
    print(f"  3' bias: {median_3prime_bias:.3f}")
    print(f"  5'/3' ratio: {median_5to3_ratio:.3f}")
    print("=" * 60)
    
    # ========== PLOTTING ==========
    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=dpi)
    
    x = np.arange(n_percentiles)
    
    # Panel 1: Coverage profile
    axes[0].fill_between(x, 0, mean_coverage, alpha=0.3, color=ALLOS_PALETTE[0])
    axes[0].plot(x, mean_coverage, color=ALLOS_PALETTE[0], linewidth=2)
    axes[0].axhline(1.0, color='gray', linestyle='--', alpha=0.7, label='Mean (1.0)')
    
    axes[0].set_xlabel("Normalized position (5' → 3')")
    axes[0].set_ylabel("Normalized coverage")
    axes[0].set_title(f"Gene Body Coverage\n(n={n_processed:,} transcripts)")
    axes[0].set_xlim(0, 100)
    axes[0].set_ylim(0, max(2.0, mean_coverage.max() * 1.1))
    axes[0].legend(fontsize=8)
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].set_axisbelow(True)
    axes[0].text(5, axes[0].get_ylim()[1]*0.95, "5'", fontsize=12, fontweight='bold')
    axes[0].text(95, axes[0].get_ylim()[1]*0.95, "3'", fontsize=12, fontweight='bold')
    
    # Panel 2: Bias bars
    bias_labels = ["5' BIAS", "3' BIAS"]
    bias_values = [median_5prime_bias, median_3prime_bias]
    colors = [ALLOS_PALETTE[0], ALLOS_PALETTE[3]]
    
    bars = axes[1].bar(bias_labels, bias_values, color=colors, edgecolor='white', linewidth=1.5)
    axes[1].axhline(1.0, color='black', linestyle='-', linewidth=1.5)
    axes[1].set_ylabel("Bias (coverage / mean)")
    axes[1].set_title("Picard Bias Metrics")
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[1].set_axisbelow(True)
    
    for bar, val in zip(bars, bias_values):
        axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03, f'{val:.3f}',
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Panel 3: Summary
    axes[2].axis('off')
    metrics_text = f"""
    Coverage Metrics
    ════════════════
    
    5' BIAS:      {median_5prime_bias:.4f}
    3' BIAS:      {median_3prime_bias:.4f}
    5'/3' RATIO:  {median_5to3_ratio:.4f}
    
    ────────────────
    Transcripts:  {n_processed:,}
    Min length:   {min_length} bp
    """
    
    axes[2].text(0.1, 0.9, metrics_text, transform=axes[2].transAxes,
                fontsize=10, fontfamily='monospace', verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='#f8f9fa', edgecolor='#dee2e6'))
    
    if median_5to3_ratio < 0.8:
        interpretation = "⚠️ 3' bias detected"
        interp_color = '#f39c12'
    elif median_5to3_ratio > 1.25:
        interpretation = "⚠️ 5' bias detected"
        interp_color = '#e74c3c'
    else:
        interpretation = "✓ Balanced coverage"
        interp_color = '#27ae60'
    
    axes[2].text(0.1, 0.25, interpretation, transform=axes[2].transAxes,
                fontsize=11, fontweight='bold', color=interp_color)
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "normalized_coverage": mean_coverage,
            "median_5prime_bias": median_5prime_bias,
            "median_3prime_bias": median_3prime_bias,
            "median_5to3_ratio": median_5to3_ratio,
            "n_transcripts": n_processed,
        }, fig
    return None





def plot_gene_coverage_single(
    bam_path: str,
    td,
    gene: str,
    *,
    transcript_id: Optional[str] = None,
    n_percentiles: int = 101,
    strand_specificity: str = "NONE",
    show_reads: bool = False,
    title: Optional[str] = None,
    figsize: Tuple[float, float] = (10, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot gene body coverage for a SINGLE gene.
    
    Useful for inspecting coverage patterns of specific genes of interest.
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file.
    td : TranscriptData
        TranscriptData object.
    gene : str
        Gene name or gene ID to plot.
    transcript_id : str, optional
        Specific transcript ID. If None, uses longest transcript for the gene.
    n_percentiles : int
        Number of percentile points (default 101).
    strand_specificity : str
        "NONE", "FIRST_READ", or "SECOND_READ".
    show_reads : bool
        If True, show individual read positions as a heatmap below coverage.
    
    Returns
    -------
    Coverage profile for the specified gene/transcript.
    """
    import pysam
    _setup_scanpy_style()
    
    # Find the gene/transcript
    target_tid = None
    target_rec = None
    
    if transcript_id:
        # Direct transcript ID lookup
        for tid, rec in td._idx._tx_by_id.items():
            if tid.split(".")[0] == transcript_id.split(".")[0]:
                target_tid = tid
                target_rec = rec
                break
    else:
        # Find by gene name - use longest transcript
        candidates = []
        for tid, rec in td._idx._tx_by_id.items():
            gene_name = rec.gene_name if hasattr(rec, 'gene_name') else ""
            gene_id = rec.gene_id if hasattr(rec, 'gene_id') else ""
            if gene.lower() in [gene_name.lower(), gene_id.split(".")[0].lower()]:
                tx_length = int(np.sum(rec.exons[:, 1] - rec.exons[:, 0])) if rec.exons.size > 0 else 0
                candidates.append((tid, rec, tx_length))
        
        if candidates:
            # Sort by length, pick longest
            candidates.sort(key=lambda x: -x[2])
            target_tid, target_rec, _ = candidates[0]
            print(f"Found {len(candidates)} transcripts for {gene}, using longest: {target_tid}")
    
    if target_rec is None:
        print(f"Gene/transcript '{gene}' not found")
        return None
    
    # Get transcript info
    chrom = target_rec.chrom
    strand = target_rec.strand
    exons = target_rec.exons
    gene_name = target_rec.gene_name if hasattr(target_rec, 'gene_name') else gene
    
    if exons.size == 0:
        print(f"No exons found for {target_tid}")
        return None
    
    tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
    exons_sorted = exons[np.argsort(exons[:, 0])]
    
    print(f"Transcript: {target_tid}")
    print(f"  Gene: {gene_name}, Chr: {chrom}, Strand: {'+' if strand == 1 else '-'}")
    print(f"  Length: {tx_length:,} bp, Exons: {len(exons_sorted)}")
    
    # Open BAM and get coverage
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM: {e}")
        return None
    
    tx_coverage = np.zeros(tx_length, dtype=np.float32)
    read_positions = []  # For optional read visualization
    
    tx_pos = 0
    for exon_start, exon_end in exons_sorted:
        exon_len = int(exon_end - exon_start)
        
        for read in bam.fetch(chrom, int(exon_start), int(exon_end)):
            if read.is_unmapped or read.is_secondary or read.is_supplementary:
                continue
            
            # Strand filtering
            if strand_specificity == "FIRST_READ":
                if read.is_read1 or not read.is_paired:
                    read_strand = -1 if read.is_reverse else 1
                    if read_strand != strand:
                        continue
            elif strand_specificity == "SECOND_READ":
                if read.is_read2 or not read.is_paired:
                    read_strand = -1 if read.is_reverse else 1
                    if read_strand != strand:
                        continue
            
            read_start = max(read.reference_start, exon_start)
            read_end = min(read.reference_end or read.reference_start + 100, exon_end)
            
            if read_end <= read_start:
                continue
            
            tx_start = tx_pos + int(read_start - exon_start)
            tx_end = tx_pos + int(read_end - exon_start)
            tx_start = max(0, min(tx_start, tx_length - 1))
            tx_end = max(0, min(tx_end, tx_length))
            
            tx_coverage[tx_start:tx_end] += 1
            
            if show_reads:
                read_positions.append((tx_start, tx_end))
        
        tx_pos += exon_len
    
    bam.close()
    
    total_reads = len(read_positions) if show_reads else int(tx_coverage.sum() / 100)  # Approximate
    print(f"  Total coverage: {tx_coverage.sum():,.0f} bases")
    
    if tx_coverage.sum() == 0:
        print("No reads found for this transcript")
        return None
    
    # Flip for minus strand
    if strand == -1:
        tx_coverage = tx_coverage[::-1]
        if show_reads:
            read_positions = [(tx_length - end, tx_length - start) for start, end in read_positions]
    
    # Normalize by mean
    mean_cov = tx_coverage.mean()
    normalized_coverage = tx_coverage / mean_cov
    
    # Sample at percentile points
    percentile_coverage = []
    last_index = tx_length - 1
    for percent in range(n_percentiles):
        p = percent / 100.0
        start_idx = int(max(0, last_index * (p - 0.005)))
        end_idx = int(min(last_index, last_index * (p + 0.005)))
        if end_idx >= start_idx:
            percentile_coverage.append(normalized_coverage[start_idx:end_idx + 1].mean())
        else:
            percentile_coverage.append(0)
    
    percentile_coverage = np.array(percentile_coverage)
    
    # Calculate bias
    five_prime_bias = normalized_coverage[:100].mean() if tx_length >= 100 else normalized_coverage.mean()
    three_prime_bias = normalized_coverage[-100:].mean() if tx_length >= 100 else normalized_coverage.mean()
    
    # ========== PLOTTING ==========
    if show_reads and read_positions:
        fig, axes = plt.subplots(2, 1, figsize=figsize, dpi=dpi, 
                                  gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
        ax_cov = axes[0]
        ax_reads = axes[1]
    else:
        fig, ax_cov = plt.subplots(figsize=figsize, dpi=dpi)
        ax_reads = None
    
    x = np.arange(n_percentiles)
    
    # Coverage plot
    ax_cov.fill_between(x, 0, percentile_coverage, alpha=0.3, color=ALLOS_PALETTE[0])
    ax_cov.plot(x, percentile_coverage, color=ALLOS_PALETTE[0], linewidth=2)
    ax_cov.axhline(1.0, color='gray', linestyle='--', alpha=0.7, label='Mean')
    
    ax_cov.set_ylabel("Normalized coverage")
    ax_cov.set_xlim(0, 100)
    ax_cov.set_ylim(0, max(2.0, percentile_coverage.max() * 1.1))
    ax_cov.legend(fontsize=8, loc='upper right')
    ax_cov.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax_cov.set_axisbelow(True)
    ax_cov.text(2, ax_cov.get_ylim()[1]*0.92, "5'", fontsize=12, fontweight='bold')
    ax_cov.text(96, ax_cov.get_ylim()[1]*0.92, "3'", fontsize=12, fontweight='bold')
    
    # Bias annotation
    bias_text = f"5' bias: {five_prime_bias:.2f}  |  3' bias: {three_prime_bias:.2f}  |  5'/3': {five_prime_bias/three_prime_bias:.2f}"
    ax_cov.text(0.5, 0.02, bias_text, transform=ax_cov.transAxes, ha='center', fontsize=9, 
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    # Read positions heatmap
    if ax_reads and read_positions:
        # Bin reads into display
        n_display_rows = min(100, len(read_positions))
        sampled_reads = read_positions[::max(1, len(read_positions)//n_display_rows)][:n_display_rows]
        
        for i, (start, end) in enumerate(sampled_reads):
            start_pct = start / tx_length * 100
            end_pct = end / tx_length * 100
            ax_reads.plot([start_pct, end_pct], [i, i], color=ALLOS_PALETTE[0], alpha=0.5, linewidth=1)
        
        ax_reads.set_ylabel("Reads")
        ax_reads.set_xlabel("Normalized position (5' → 3')")
        ax_reads.set_ylim(-1, n_display_rows)
        ax_reads.set_yticks([])
    else:
        ax_cov.set_xlabel("Normalized position (5' → 3')")
    
    if title is None:
        title = f"{gene_name} ({target_tid.split('.')[0]})"
    
    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "transcript_id": target_tid,
            "gene": gene_name,
            "length": tx_length,
            "raw_coverage": tx_coverage * mean_cov,  # Un-normalize
            "normalized_coverage": normalized_coverage,
            "percentile_coverage": percentile_coverage,
            "five_prime_bias": five_prime_bias,
            "three_prime_bias": three_prime_bias,
            "strand": strand
        }, fig
    return None




def plot_read_start_positions(
    bam_path: str,
    td,
    *,
    n_bins: int = 100,
    max_transcripts: int = 2000,
    min_reads: int = 10,
    min_length: int = 500,
    title: str = "Read Start Position Distribution (5' → 3')",
    figsize: Tuple[float, float] = (10, 6),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot where reads START along the transcript body (5' to 3').
    
    Unlike coverage (which is flattened by long reads), read start positions
    reveal the true 3' bias from poly-A priming.
    
    For poly-A primed single-cell data, expect reads to START near the 3' end.
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM file: {e}")
        return None
    
    start_positions = []  # Normalized 0-1 position where each read starts
    n_processed = 0
    
    tx_items = list(td._idx._tx_by_id.items())
    np.random.shuffle(tx_items)
    
    for tid, rec in tx_items:
        if n_processed >= max_transcripts:
            break
        
        chrom = rec.chrom
        strand = rec.strand  # 1 or -1
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        if tx_length < min_length:
            continue
        
        exons_sorted = exons[np.argsort(exons[:, 0])]
        
        # Build cumulative exon positions for mapping
        exon_cum_start = []
        cum = 0
        for exon_start, exon_end in exons_sorted:
            exon_cum_start.append((exon_start, exon_end, cum))
            cum += (exon_end - exon_start)
        
        tx_read_starts = []
        
        try:
            for exon_start, exon_end in exons_sorted:
                for read in bam.fetch(chrom, exon_start, exon_end):
                    if read.is_unmapped or read.is_secondary or read.is_supplementary:
                        continue
                    
                    # Get read start in genomic coordinates
                    # For forward reads: reference_start is the 5' end of the read
                    # For reverse reads: reference_end is the 5' end of the read
                    if read.is_reverse:
                        read_5prime = read.reference_end - 1 if read.reference_end else read.reference_start
                    else:
                        read_5prime = read.reference_start
                    
                    # Check if this start is within the current exon
                    if exon_start <= read_5prime < exon_end:
                        # Map to transcript coordinates
                        for ex_start, ex_end, cum_pos in exon_cum_start:
                            if ex_start <= read_5prime < ex_end:
                                tx_pos = cum_pos + (read_5prime - ex_start)
                                break
                        else:
                            continue
                        
                        # Normalize to 0-1 (5' to 3')
                        if strand == -1:
                            # Minus strand: flip so 5' is at 0
                            norm_pos = 1.0 - (tx_pos / tx_length)
                        else:
                            norm_pos = tx_pos / tx_length
                        
                        tx_read_starts.append(norm_pos)
            
            if len(tx_read_starts) >= min_reads:
                start_positions.extend(tx_read_starts)
                n_processed += 1
                
                if n_processed % 500 == 0:
                    print(f"  Processed {n_processed} transcripts, {len(start_positions)} reads...")
        
        except Exception as e:
            continue
    
    bam.close()
    
    print(f"Processed {n_processed} transcripts, {len(start_positions):,} read starts")
    
    if len(start_positions) == 0:
        print("No reads found.")
        return None
    
    start_positions = np.array(start_positions)
    
    # Bin and plot
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=dpi)
    
    # Panel 1: Histogram of read start positions
    counts, bin_edges = np.histogram(start_positions, bins=n_bins, range=(0, 1))
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2 * 100
    
    # Normalize
    counts_norm = counts / counts.max()
    
    axes[0].fill_between(bin_centers, 0, counts_norm, alpha=0.3, color=ALLOS_PALETTE[0])
    axes[0].plot(bin_centers, counts_norm, color=ALLOS_PALETTE[0], linewidth=2)
    axes[0].axhline(counts_norm.mean(), color='gray', linestyle='--', alpha=0.5, label='Mean')
    
    axes[0].set_xlabel("Transcript position (5' → 3')")
    axes[0].set_ylabel("Read starts (normalized)")
    axes[0].set_title(f"Read Start Positions\n(n={len(start_positions):,} reads)")
    axes[0].set_xlim(0, 100)
    axes[0].legend(fontsize=8)
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].text(5, axes[0].get_ylim()[1]*0.95, "5'", fontsize=12, fontweight='bold')
    axes[0].text(95, axes[0].get_ylim()[1]*0.95, "3'", fontsize=12, fontweight='bold')
    
    # Panel 2: Cumulative distribution
    sorted_pos = np.sort(start_positions) * 100
    cdf = np.arange(1, len(sorted_pos) + 1) / len(sorted_pos)
    
    axes[1].plot(sorted_pos, cdf, color=ALLOS_PALETTE[0], linewidth=2, label='Observed')
    axes[1].plot([0, 100], [0, 1], 'k--', alpha=0.5, label='Uniform (no bias)')
    
    # Calculate bias metrics
    pct_in_3prime_half = (start_positions > 0.5).mean() * 100
    pct_in_3prime_quarter = (start_positions > 0.75).mean() * 100
    median_pos = np.median(start_positions) * 100
    
    axes[1].set_xlabel("Transcript position (5' → 3')")
    axes[1].set_ylabel("Cumulative fraction of reads")
    axes[1].set_title("Cumulative Distribution")
    axes[1].legend(fontsize=8)
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    
    # Stats box
    stats_text = (f"Median start: {median_pos:.1f}%\n"
                  f"Starts in 3\' half: {pct_in_3prime_half:.1f}%\n"
                  f"Starts in 3\' quarter: {pct_in_3prime_quarter:.1f}%")
    axes[1].text(0.02, 0.98, stats_text, transform=axes[1].transAxes,
                ha='left', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    # Verdict
    if pct_in_3prime_half > 60:
        verdict = f"3\' bias detected: {pct_in_3prime_half:.0f}% of reads start in 3\' half"
        verdict_color = '#f39c12'
    elif pct_in_3prime_half < 40:
        verdict = f"5\' bias detected: {100-pct_in_3prime_half:.0f}% of reads start in 5\' half"
        verdict_color = '#e74c3c'
    else:
        verdict = f"Balanced: {pct_in_3prime_half:.0f}% in 3\' half (expected ~50%)"
        verdict_color = '#27ae60'
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    fig.text(0.5, -0.02, verdict, ha='center', fontsize=11, fontweight='bold', color=verdict_color)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "start_positions": start_positions,
            "median_position": median_pos,
            "pct_3prime_half": pct_in_3prime_half,
            "n_reads": len(start_positions)
        }, fig
    return None





# ==================== LONG-READ SINGLE-CELL QC FUNCTIONS ====================

def plot_tss_enrichment(
    bam_path: str,
    td,
    *,
    window: int = 2000,
    n_bins: int = 100,
    max_transcripts: int = 5000,
    min_length: int = 500,
    title: str = "TSS Enrichment",
    figsize: Tuple[float, float] = (8, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Calculate TSS (Transcription Start Site) enrichment score.
    
    For long-read data, this measures how well reads capture the true 5' end.
    High TSS enrichment = good full-length capture.
    
    TSS enrichment score (ENCODE definition):
        Signal at TSS / Signal at flanks
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file.
    td : TranscriptData
        TranscriptData object.
    window : int
        Window size around TSS (default ±2000bp).
    n_bins : int
        Number of bins for the profile.
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM: {e}")
        return None
    
    # Aggregate signal around TSS
    tss_profile = np.zeros(n_bins * 2)  # -window to +window
    n_transcripts = 0
    
    tx_items = list(td._idx._tx_by_id.items())
    np.random.shuffle(tx_items)
    
    for tid, rec in tx_items:
        if n_transcripts >= max_transcripts:
            break
        
        chrom = rec.chrom
        strand = rec.strand
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        if tx_length < min_length:
            continue
        
        # Get TSS position (5' end of transcript)
        if strand == 1:
            tss = int(exons.min())  # Leftmost position for + strand
        else:
            tss = int(exons.max())  # Rightmost position for - strand
        
        # Fetch reads around TSS
        start = max(0, tss - window)
        end = tss + window
        
        local_profile = np.zeros(n_bins * 2)
        bin_size = (2 * window) / (n_bins * 2)
        
        try:
            for read in bam.fetch(chrom, start, end):
                if read.is_unmapped or read.is_secondary or read.is_supplementary:
                    continue
                
                # Get read 5' end position
                if read.is_reverse:
                    read_5prime = read.reference_end - 1 if read.reference_end else read.reference_start
                else:
                    read_5prime = read.reference_start
                
                # Position relative to TSS
                if strand == 1:
                    rel_pos = read_5prime - tss
                else:
                    rel_pos = tss - read_5prime  # Flip for minus strand
                
                if -window <= rel_pos < window:
                    bin_idx = int((rel_pos + window) / bin_size)
                    bin_idx = max(0, min(bin_idx, n_bins * 2 - 1))
                    local_profile[bin_idx] += 1
            
            if local_profile.sum() > 0:
                tss_profile += local_profile / local_profile.sum()  # Normalize per transcript
                n_transcripts += 1
        except:
            continue
    
    bam.close()
    
    if n_transcripts == 0:
        print("No transcripts with reads found")
        return None
    
    tss_profile /= n_transcripts
    
    # Calculate TSS enrichment score (ENCODE style)
    # Signal at center / signal at flanks
    center_signal = tss_profile[n_bins - 10:n_bins + 10].mean()
    flank_signal = (tss_profile[:10].mean() + tss_profile[-10:].mean()) / 2
    tss_enrichment = center_signal / flank_signal if flank_signal > 0 else 0
    
    print(f"Analyzed {n_transcripts:,} transcripts")
    print(f"TSS Enrichment Score: {tss_enrichment:.2f}")
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    x = np.linspace(-window, window, n_bins * 2)
    ax.fill_between(x, 0, tss_profile, alpha=0.3, color=ALLOS_PALETTE[0])
    ax.plot(x, tss_profile, color=ALLOS_PALETTE[0], linewidth=2)
    ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='TSS')
    
    ax.set_xlabel("Distance from TSS (bp)")
    ax.set_ylabel("Normalized read density")
    ax.set_title(f"{title}\n(n={n_transcripts:,} transcripts)")
    ax.legend(fontsize=9)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Add enrichment score
    ax.text(0.98, 0.98, f"TSS Enrichment: {tss_enrichment:.2f}", 
           transform=ax.transAxes, ha='right', va='top', fontsize=11, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {"profile": tss_profile, "enrichment": tss_enrichment, "n_transcripts": n_transcripts}, fig
    return None


def plot_tes_enrichment(
    bam_path: str,
    td,
    *,
    window: int = 2000,
    n_bins: int = 100,
    max_transcripts: int = 5000,
    min_length: int = 500,
    title: str = "TES Enrichment (Poly-A Site)",
    figsize: Tuple[float, float] = (8, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Calculate TES (Transcription End Site / Poly-A site) enrichment.
    
    For poly-A primed single-cell data, reads should be enriched at TES.
    This measures how well the 3' end is captured.
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file.
    td : TranscriptData
        TranscriptData object.
    window : int
        Window size around TES (default ±2000bp).
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM: {e}")
        return None
    
    tes_profile = np.zeros(n_bins * 2)
    n_transcripts = 0
    
    tx_items = list(td._idx._tx_by_id.items())
    np.random.shuffle(tx_items)
    
    for tid, rec in tx_items:
        if n_transcripts >= max_transcripts:
            break
        
        chrom = rec.chrom
        strand = rec.strand
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        if tx_length < min_length:
            continue
        
        # Get TES position (3' end / poly-A site)
        if strand == 1:
            tes = int(exons.max())  # Rightmost for + strand
        else:
            tes = int(exons.min())  # Leftmost for - strand
        
        start = max(0, tes - window)
        end = tes + window
        
        local_profile = np.zeros(n_bins * 2)
        bin_size = (2 * window) / (n_bins * 2)
        
        try:
            for read in bam.fetch(chrom, start, end):
                if read.is_unmapped or read.is_secondary or read.is_supplementary:
                    continue
                
                # Get read 3' end position (where cDNA synthesis started for poly-A priming)
                if read.is_reverse:
                    read_3prime = read.reference_start
                else:
                    read_3prime = read.reference_end - 1 if read.reference_end else read.reference_start
                
                # Position relative to TES
                if strand == 1:
                    rel_pos = read_3prime - tes
                else:
                    rel_pos = tes - read_3prime
                
                if -window <= rel_pos < window:
                    bin_idx = int((rel_pos + window) / bin_size)
                    bin_idx = max(0, min(bin_idx, n_bins * 2 - 1))
                    local_profile[bin_idx] += 1
            
            if local_profile.sum() > 0:
                tes_profile += local_profile / local_profile.sum()
                n_transcripts += 1
        except:
            continue
    
    bam.close()
    
    if n_transcripts == 0:
        print("No transcripts with reads found")
        return None
    
    tes_profile /= n_transcripts
    
    center_signal = tes_profile[n_bins - 10:n_bins + 10].mean()
    flank_signal = (tes_profile[:10].mean() + tes_profile[-10:].mean()) / 2
    tes_enrichment = center_signal / flank_signal if flank_signal > 0 else 0
    
    print(f"Analyzed {n_transcripts:,} transcripts")
    print(f"TES Enrichment Score: {tes_enrichment:.2f}")
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    x = np.linspace(-window, window, n_bins * 2)
    ax.fill_between(x, 0, tes_profile, alpha=0.3, color=ALLOS_PALETTE[3])
    ax.plot(x, tes_profile, color=ALLOS_PALETTE[3], linewidth=2)
    ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='TES (Poly-A)')
    
    ax.set_xlabel("Distance from TES (bp)")
    ax.set_ylabel("Normalized read density")
    ax.set_title(f"{title}\n(n={n_transcripts:,} transcripts)")
    ax.legend(fontsize=9)
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    ax.text(0.98, 0.98, f"TES Enrichment: {tes_enrichment:.2f}",
           transform=ax.transAxes, ha='right', va='top', fontsize=11, fontweight='bold',
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {"profile": tes_profile, "enrichment": tes_enrichment, "n_transcripts": n_transcripts}, fig
    return None


def plot_full_length_ratio(
    bam_path: str,
    td,
    *,
    tss_tolerance: int = 50,
    tes_tolerance: int = 50,
    max_transcripts: int = 5000,
    min_length: int = 500,
    min_reads: int = 5,
    title: str = "Full-Length Transcript Capture",
    figsize: Tuple[float, float] = (10, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Calculate the percentage of reads that are full-length (TSS to TES).
    
    This is THE key metric for long-read RNA-seq - what fraction of reads
    span from the annotated TSS to TES?
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file.
    td : TranscriptData
        TranscriptData object.
    tss_tolerance : int
        How close to annotated TSS to count as "reaching TSS" (default 50bp).
    tes_tolerance : int
        How close to annotated TES to count as "reaching TES" (default 50bp).
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM: {e}")
        return None
    
    results = []  # Per-transcript results
    
    tx_items = list(td._idx._tx_by_id.items())
    np.random.shuffle(tx_items)
    n_processed = 0
    
    for tid, rec in tx_items:
        if n_processed >= max_transcripts:
            break
        
        chrom = rec.chrom
        strand = rec.strand
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        if tx_length < min_length:
            continue
        
        # Get TSS and TES
        if strand == 1:
            tss = int(exons.min())
            tes = int(exons.max())
        else:
            tss = int(exons.max())
            tes = int(exons.min())
        
        # Count reads by category
        n_full_length = 0
        n_has_tss = 0
        n_has_tes = 0
        n_total = 0
        
        try:
            for read in bam.fetch(chrom, min(tss, tes) - 100, max(tss, tes) + 100):
                if read.is_unmapped or read.is_secondary or read.is_supplementary:
                    continue
                
                read_start = read.reference_start
                read_end = read.reference_end or read_start + 100
                
                # Check if read reaches TSS
                if strand == 1:
                    reaches_tss = read_start <= tss + tss_tolerance
                    reaches_tes = read_end >= tes - tes_tolerance
                else:
                    reaches_tss = read_end >= tss - tss_tolerance
                    reaches_tes = read_start <= tes + tes_tolerance
                
                n_total += 1
                if reaches_tss:
                    n_has_tss += 1
                if reaches_tes:
                    n_has_tes += 1
                if reaches_tss and reaches_tes:
                    n_full_length += 1
            
            if n_total >= min_reads:
                results.append({
                    "tid": tid,
                    "length": tx_length,
                    "n_reads": n_total,
                    "n_full_length": n_full_length,
                    "n_has_tss": n_has_tss,
                    "n_has_tes": n_has_tes,
                    "pct_full_length": n_full_length / n_total * 100,
                    "pct_has_tss": n_has_tss / n_total * 100,
                    "pct_has_tes": n_has_tes / n_total * 100
                })
                n_processed += 1
        except:
            continue
    
    bam.close()
    
    if not results:
        print("No transcripts with sufficient reads found")
        return None
    
    df = pd.DataFrame(results)
    
    # Summary statistics
    overall_full_length = df["n_full_length"].sum() / df["n_reads"].sum() * 100
    overall_has_tss = df["n_has_tss"].sum() / df["n_reads"].sum() * 100
    overall_has_tes = df["n_has_tes"].sum() / df["n_reads"].sum() * 100
    
    print(f"Analyzed {len(df):,} transcripts, {df['n_reads'].sum():,} reads")
    print(f"  Reads reaching TSS: {overall_has_tss:.1f}%")
    print(f"  Reads reaching TES: {overall_has_tes:.1f}%")
    print(f"  Full-length reads: {overall_full_length:.1f}%")
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=figsize, dpi=dpi)
    
    # Panel 1: Bar chart of categories
    categories = ["Reaches\nTSS (5')", "Reaches\nTES (3')", "Full-length\n(TSS→TES)"]
    values = [overall_has_tss, overall_has_tes, overall_full_length]
    colors = [ALLOS_PALETTE[0], ALLOS_PALETTE[3], ALLOS_PALETTE[2]]
    
    bars = axes[0].bar(categories, values, color=colors, edgecolor='white', linewidth=1.5)
    axes[0].set_ylabel("Percentage of reads")
    axes[0].set_title("Read Coverage Summary")
    axes[0].set_ylim(0, 100)
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    
    for bar, val in zip(bars, values):
        axes[0].text(bar.get_x() + bar.get_width()/2, val + 2, f'{val:.1f}%',
                    ha='center', fontsize=10, fontweight='bold')
    
    # Panel 2: Full-length % vs transcript length
    axes[1].scatter(df["length"], df["pct_full_length"], alpha=0.5, s=20, 
                   c=ALLOS_PALETTE[2], edgecolors='none')
    
    # Add trend line
    z = np.polyfit(np.log10(df["length"]), df["pct_full_length"], 1)
    x_line = np.linspace(df["length"].min(), df["length"].max(), 100)
    axes[1].plot(x_line, z[0] * np.log10(x_line) + z[1], 'r--', alpha=0.7, linewidth=2)
    
    axes[1].set_xlabel("Transcript length (bp)")
    axes[1].set_ylabel("% Full-length reads")
    axes[1].set_title("Full-length vs Length")
    axes[1].set_xscale('log')
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    
    # Panel 3: Distribution of full-length %
    axes[2].hist(df["pct_full_length"], bins=30, color=ALLOS_PALETTE[2], 
                edgecolor='white', alpha=0.7)
    axes[2].axvline(df["pct_full_length"].median(), color='red', linestyle='--', 
                   label=f'Median: {df["pct_full_length"].median():.1f}%')
    axes[2].set_xlabel("% Full-length per transcript")
    axes[2].set_ylabel("Number of transcripts")
    axes[2].set_title("Distribution")
    axes[2].legend(fontsize=9)
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "per_transcript": df,
            "overall_full_length_pct": overall_full_length,
            "overall_tss_pct": overall_has_tss,
            "overall_tes_pct": overall_has_tes
        }, fig
    return None


def plot_read_length_vs_position(
    bam_path: str,
    td,
    *,
    max_transcripts: int = 2000,
    min_length: int = 500,
    min_reads: int = 10,
    title: str = "Read Length vs Transcript Position",
    figsize: Tuple[float, float] = (10, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Analyze correlation between read length and position along transcript.
    
    Reveals if truncated reads are biased to certain positions:
    - Short reads at 3' = RT fell off early (5' dropout)
    - Short reads uniform = random degradation
    - Long reads at 3' = good full-length capture
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file.
    td : TranscriptData
        TranscriptData object.
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM: {e}")
        return None
    
    read_data = []  # (normalized_start_position, read_length, transcript_length)
    
    tx_items = list(td._idx._tx_by_id.items())
    np.random.shuffle(tx_items)
    n_processed = 0
    
    for tid, rec in tx_items:
        if n_processed >= max_transcripts:
            break
        
        chrom = rec.chrom
        strand = rec.strand
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        if tx_length < min_length:
            continue
        
        exons_sorted = exons[np.argsort(exons[:, 0])]
        tx_start_genomic = int(exons_sorted[0, 0])
        tx_end_genomic = int(exons_sorted[-1, 1])
        
        n_reads = 0
        try:
            for read in bam.fetch(chrom, tx_start_genomic, tx_end_genomic):
                if read.is_unmapped or read.is_secondary or read.is_supplementary:
                    continue
                
                read_len = read.query_length or (read.reference_end - read.reference_start if read.reference_end else 100)
                
                # Get read start position relative to transcript
                if strand == 1:
                    rel_start = (read.reference_start - tx_start_genomic) / tx_length
                else:
                    rel_start = (tx_end_genomic - (read.reference_end or read.reference_start)) / tx_length
                
                rel_start = max(0, min(1, rel_start))
                
                read_data.append({
                    "position": rel_start * 100,  # 0-100%
                    "read_length": read_len,
                    "tx_length": tx_length
                })
                n_reads += 1
            
            if n_reads >= min_reads:
                n_processed += 1
        except:
            continue
    
    bam.close()
    
    if not read_data:
        print("No reads found")
        return None
    
    df = pd.DataFrame(read_data)
    
    print(f"Analyzed {len(df):,} reads from {n_processed} transcripts")
    print(f"  Mean read length: {df['read_length'].mean():.0f} bp")
    print(f"  Median read length: {df['read_length'].median():.0f} bp")
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=dpi)
    
    # Panel 1: 2D histogram of position vs read length
    h = axes[0].hexbin(df["position"], df["read_length"], gridsize=40, cmap="Blues", mincnt=1)
    plt.colorbar(h, ax=axes[0], label="Read count")
    
    # Add trend line
    from scipy import stats
    slope, intercept, r, p, se = stats.linregress(df["position"], df["read_length"])
    x_line = np.array([0, 100])
    axes[0].plot(x_line, slope * x_line + intercept, 'r--', linewidth=2, 
                label=f'r={r:.3f}, p={p:.2e}')
    
    axes[0].set_xlabel("Position along transcript (5' → 3')")
    axes[0].set_ylabel("Read length (bp)")
    axes[0].set_title("Read Length vs Position")
    axes[0].legend(fontsize=9)
    axes[0].text(5, axes[0].get_ylim()[1]*0.95, "5'", fontsize=11, fontweight='bold')
    axes[0].text(95, axes[0].get_ylim()[1]*0.95, "3'", fontsize=11, fontweight='bold')
    
    # Panel 2: Mean read length by position bins
    df["pos_bin"] = pd.cut(df["position"], bins=20, labels=range(20))
    mean_by_bin = df.groupby("pos_bin")["read_length"].mean()
    
    axes[1].bar(range(20), mean_by_bin.values, color=ALLOS_PALETTE[0], 
               edgecolor='white', linewidth=0.5)
    axes[1].axhline(df["read_length"].mean(), color='gray', linestyle='--', 
                   label=f'Overall mean: {df["read_length"].mean():.0f} bp')
    
    axes[1].set_xlabel("Position along transcript (5' → 3')")
    axes[1].set_ylabel("Mean read length (bp)")
    axes[1].set_title("Mean Read Length by Position")
    axes[1].set_xticks([0, 10, 19])
    axes[1].set_xticklabels(["5'", "middle", "3'"])
    axes[1].legend(fontsize=9)
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {"reads": df, "correlation": r, "p_value": p}, fig
    return None


def plot_coverage_by_transcript_length(
    bam_path: str,
    td,
    *,
    length_bins: List[Tuple[int, int]] = [(0, 1000), (1000, 2000), (2000, 5000), (5000, 100000)],
    n_percentiles: int = 101,
    max_per_bin: int = 500,
    min_reads: int = 10,
    title: str = "Coverage by Transcript Length",
    figsize: Tuple[float, float] = (10, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Gene body coverage stratified by transcript length.
    
    OPTIMIZED: Uses count_coverage for fast pre-filtering.
    
    LRGASP found: "coverage strongly dropped towards the 5' end for the longest molecules"
    This reveals if longer transcripts have worse 5' capture.
    
    Parameters
    ----------
    bam_path : str
        Path to BAM file.
    td : TranscriptData
        TranscriptData object.
    length_bins : list of tuples
        Transcript length bins as (min, max) pairs.
    """
    import pysam
    _setup_scanpy_style()
    
    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
    except Exception as e:
        print(f"Error opening BAM: {e}")
        return None
    
    # ========== STEP 1: Fast pre-filtering with count_coverage ==========
    print("Step 1: Fast pre-filtering transcripts by length bin...")
    
    # Organize candidates by length bin
    bin_candidates = {i: [] for i in range(len(length_bins))}
    n_checked = 0
    
    for tid, rec in td._idx._tx_by_id.items():
        chrom = rec.chrom
        exons = rec.exons
        
        if exons.size == 0:
            continue
        
        tx_length = int(np.sum(exons[:, 1] - exons[:, 0]))
        
        # Find which bin this transcript belongs to
        bin_idx = None
        for i, (min_len, max_len) in enumerate(length_bins):
            if min_len <= tx_length < max_len:
                bin_idx = i
                break
        
        if bin_idx is None:
            continue
        
        # Skip if this bin already has enough candidates
        if len(bin_candidates[bin_idx]) >= max_per_bin * 2:  # Get 2x for filtering
            continue
        
        # Fast coverage check
        total_depth = 0
        try:
            for exon_start, exon_end in exons:
                exon_start, exon_end = int(exon_start), int(exon_end)
                cov_tuple = bam.count_coverage(chrom, exon_start, exon_end,
                                               quality_threshold=0,
                                               read_callback='nofilter')
                for arr in cov_tuple:
                    total_depth += sum(arr)
        except:
            continue
        
        if total_depth >= min_reads:
            bin_candidates[bin_idx].append((tid, total_depth, tx_length, rec.strand, exons, chrom))
        
        n_checked += 1
        if n_checked % 5000 == 0:
            counts = [len(bin_candidates[i]) for i in range(len(length_bins))]
            print(f"  Checked {n_checked}, candidates per bin: {counts}")
    
    # Report candidates per bin
    for i, (min_len, max_len) in enumerate(length_bins):
        print(f"  Bin {min_len}-{max_len}bp: {len(bin_candidates[i])} candidates")
    
    # ========== STEP 2: Process candidates in each bin ==========
    print("Step 2: Computing coverage profiles per bin...")
    
    bin_profiles = {i: [] for i in range(len(length_bins))}
    bin_counts = {i: 0 for i in range(len(length_bins))}
    
    for bin_idx in range(len(length_bins)):
        candidates = bin_candidates[bin_idx]
        np.random.shuffle(candidates)
        
        for tid, total_depth, tx_length, strand, exons, chrom in candidates[:max_per_bin]:
            exons_sorted = exons[np.argsort(exons[:, 0])]
            tx_coverage = np.zeros(tx_length, dtype=np.float32)
            
            try:
                tx_pos = 0
                for exon_start, exon_end in exons_sorted:
                    exon_start, exon_end = int(exon_start), int(exon_end)
                    exon_len = exon_end - exon_start
                    
                    for read in bam.fetch(chrom, exon_start, exon_end):
                        if read.is_unmapped or read.is_secondary or read.is_supplementary:
                            continue
                        
                        read_start = max(read.reference_start, exon_start)
                        read_end = min(read.reference_end or read.reference_start + 100, exon_end)
                        
                        if read_end <= read_start:
                            continue
                        
                        tx_start = tx_pos + int(read_start - exon_start)
                        tx_end = tx_pos + int(read_end - exon_start)
                        tx_start = max(0, min(tx_start, tx_length - 1))
                        tx_end = max(0, min(tx_end, tx_length))
                        
                        tx_coverage[tx_start:tx_end] += 1
                    
                    tx_pos += exon_len
                
                if tx_coverage.sum() < min_reads:
                    continue
                
                # Flip for minus strand
                if strand == -1:
                    tx_coverage = tx_coverage[::-1]
                
                # Normalize
                mean_cov = tx_coverage.mean()
                if mean_cov == 0:
                    continue
                
                # Sample at percentile points
                profile = []
                last_index = tx_length - 1
                for percent in range(n_percentiles):
                    p = percent / 100.0
                    start_idx = int(max(0, last_index * (p - 0.005)))
                    end_idx = int(min(last_index, last_index * (p + 0.005)))
                    if end_idx >= start_idx:
                        profile.append(tx_coverage[start_idx:end_idx + 1].mean() / mean_cov)
                    else:
                        profile.append(0)
                
                bin_profiles[bin_idx].append(profile)
                bin_counts[bin_idx] += 1
                
            except:
                continue
    
    bam.close()
    
    # Calculate mean profiles per bin
    mean_profiles = {}
    for i, (min_len, max_len) in enumerate(length_bins):
        if bin_profiles[i]:
            mean_profiles[i] = np.mean(bin_profiles[i], axis=0)
            print(f"  {min_len}-{max_len}bp: {len(bin_profiles[i])} transcripts")
    
    if not mean_profiles:
        print("No transcripts found")
        return None
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=dpi)
    
    x = np.arange(n_percentiles)
    colors = ALLOS_PALETTE[:len(length_bins)]
    
    # Panel 1: Overlaid coverage profiles
    for i, (min_len, max_len) in enumerate(length_bins):
        if i in mean_profiles:
            label = f"{min_len/1000:.0f}-{max_len/1000:.0f}kb" if max_len < 100000 else f">{min_len/1000:.0f}kb"
            axes[0].plot(x, mean_profiles[i], linewidth=2, color=colors[i], label=label)
    
    axes[0].axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_xlabel("Normalized position (5' → 3')")
    axes[0].set_ylabel("Normalized coverage")
    axes[0].set_title("Coverage by Transcript Length")
    axes[0].set_xlim(0, 100)
    axes[0].legend(fontsize=9, title="Length")
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].text(5, axes[0].get_ylim()[1]*0.95, "5'", fontsize=11, fontweight='bold')
    axes[0].text(95, axes[0].get_ylim()[1]*0.95, "3'", fontsize=11, fontweight='bold')
    
    # Panel 2: 5'/3' ratio by length bin
    ratios = []
    labels = []
    for i, (min_len, max_len) in enumerate(length_bins):
        if i in mean_profiles:
            five_prime = mean_profiles[i][:20].mean()
            three_prime = mean_profiles[i][-20:].mean()
            ratio = five_prime / three_prime if three_prime > 0 else 0
            ratios.append(ratio)
            label = f"{min_len/1000:.0f}-{max_len/1000:.0f}kb" if max_len < 100000 else f">{min_len/1000:.0f}kb"
            labels.append(label)
    
    bar_colors = ['#27ae60' if r > 0.8 else '#f39c12' if r > 0.5 else '#e74c3c' for r in ratios]
    bars = axes[1].bar(labels, ratios, color=bar_colors, edgecolor='white', linewidth=1.5)
    axes[1].axhline(1.0, color='black', linestyle='-', linewidth=1.5)
    axes[1].axhline(0.8, color='gray', linestyle='--', alpha=0.5)
    
    axes[1].set_ylabel("5'/3' Coverage Ratio")
    axes[1].set_title("5' Dropout by Length")
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.3)
    
    for bar, ratio in zip(bars, ratios):
        axes[1].text(bar.get_x() + bar.get_width()/2, ratio + 0.03, f'{ratio:.2f}',
                    ha='center', fontsize=10, fontweight='bold')
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return {
            "mean_profiles": mean_profiles,
            "bin_counts": bin_counts,
            "ratios": dict(zip(labels, ratios))
        }, fig
    return None



def plot_length_by_biotype(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    biotype_col: str = "transcript_type",  # column in adata.var or from GTF
    top_n_biotypes: int = 8,
    log_scale: bool = True,
    show_counts: bool = True,
    title: str = "Transcript Length by Biotype",
    figsize: Optional[Tuple[float, float]] = None,
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Plot transcript length distribution grouped by biotype (protein_coding, lncRNA, etc.).
    
    Biotypes are retrieved from TranscriptData (GTF annotation).
    """
    import seaborn as sns
    _setup_scanpy_style()
    
    # Get transcript info including biotype from TranscriptIndex
    biotype_data = []
    for tid, rec in td._idx._tx_by_id.items():
        tid_stripped = tid.split(".")[0]
        biotype = rec.transcript_type if rec.transcript_type and rec.transcript_type != "nan" else "unknown"
        length = int(np.sum(rec.exons[:, 1] - rec.exons[:, 0])) if rec.exons.size > 0 else 0
        biotype_data.append({
            "transcript_id": tid_stripped,
            "biotype": biotype,
            "length": length
        })
    
    biotype_df = pd.DataFrame(biotype_data)
    
    # Filter to transcripts in adata
    if adata_tx_col is None:
        adata_tx = set(_strip_ver(adata.var.index))
    else:
        adata_tx = set(_strip_ver(adata.var[adata_tx_col].values))
    
    df = biotype_df[biotype_df["transcript_id"].isin(adata_tx)].copy()
    
    if df.empty:
        print("No matching transcripts found.")
        return None
    
    # Get top biotypes
    top_biotypes = df["biotype"].value_counts().head(top_n_biotypes).index.tolist()
    df["biotype_plot"] = df["biotype"].apply(lambda x: x if x in top_biotypes else "other")
    
    # Order by median length
    order = df.groupby("biotype_plot")["length"].median().sort_values(ascending=False).index.tolist()
    
    if log_scale:
        df["length_plot"] = np.log10(df["length"].clip(lower=1))
        length_label = "Transcript length (log₁₀ bp)"
    else:
        df["length_plot"] = df["length"]
        length_label = "Transcript length (bp)"
    
    n_biotypes = df["biotype_plot"].nunique()
    if figsize is None:
        figsize = (max(8, n_biotypes * 0.8 + 2), 6)
    
    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    
    sns.violinplot(data=df, x="biotype_plot", y="length_plot", order=order,
                  palette=ALLOS_PALETTE[:n_biotypes], ax=ax,
                  inner="box", linewidth=1, cut=0, density_norm="width")
    
    ax.set_xlabel("")
    ax.set_ylabel(length_label)
    ax.set_title(title, pad=15)
    plt.xticks(rotation=45, ha='right')
    ax.yaxis.grid(True, linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    
    # Add counts
    if show_counts:
        counts = df["biotype_plot"].value_counts().reindex(order)
        for i, (biotype, count) in enumerate(counts.items()):
            ax.text(i, ax.get_ylim()[1] * 0.98, f'n={count:,}',
                   ha='center', va='top', fontsize=8, color='#666666')
    
    # Add reference lines
    if log_scale:
        for size in [1000, 5000, 10000]:
            if np.log10(size) < ax.get_ylim()[1]:
                ax.axhline(np.log10(size), color='gray', linestyle=':', alpha=0.4)
    
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return df, fig
    return None


def plot_length_by_annotation_status(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    layer: Optional[str] = None,
    log_scale: bool = True,
    title: str = "Transcript Length: Annotated vs Novel",
    figsize: Tuple[float, float] = (10, 5),
    dpi: int = 150,
    save: Optional[str] = None,
    show: bool = True,
    return_data: bool = False,
):
    """
    Compare length distributions between annotated and potentially novel transcripts.
    
    Transcripts in adata that match GTF annotations are "annotated".
    Those without GTF match may be novel isoforms.
    """
    import seaborn as sns
    _setup_scanpy_style()
    
    # Get all annotated transcript IDs from TranscriptData
    annotated_ids = set()
    for tid in td._idx._tx_by_id.keys():
        annotated_ids.add(tid.split(".")[0])
    
    # Get transcripts from adata
    if adata_tx_col is None:
        adata_tx = list(_strip_ver(adata.var.index))
    else:
        adata_tx = list(_strip_ver(adata.var[adata_tx_col].values))
    
    # Get lengths from GTF
    length_table = td.transcript_lengths_table()
    length_dict = dict(zip(length_table["transcript_id"], length_table["length"]))
    
    # Classify and get lengths
    # For transcripts not in GTF, we'd need read-level data for actual length
    # Here we can only show GTF-annotated ones with their expected lengths
    X, _ = _get_matrix(adata, layer)
    mean_expr = np.asarray(X.mean(axis=0)).ravel() if sparse.issparse(X) else X.mean(axis=0)
    
    data = []
    for i, tx in enumerate(adata_tx):
        status = "Annotated" if tx in annotated_ids else "Novel/Unannotated"
        length = length_dict.get(tx, 0)
        if length > 0:  # Only include if we have length info
            data.append({
                "transcript_id": tx,
                "status": status,
                "length": length,
                "mean_expression": mean_expr[i]
            })
    
    df = pd.DataFrame(data)
    
    if df.empty:
        print("No transcripts with length information found.")
        return None
    
    if log_scale:
        df["length_plot"] = np.log10(df["length"].clip(lower=1))
        length_label = "Transcript length (log₁₀ bp)"
    else:
        df["length_plot"] = df["length"]
        length_label = "Transcript length (bp)"
    
    fig, axes = plt.subplots(1, 2, figsize=figsize, dpi=dpi)
    
    # Left: Length distribution by status
    colors = {"Annotated": ALLOS_PALETTE[0], "Novel/Unannotated": ALLOS_PALETTE[3]}
    
    sns.violinplot(data=df, x="status", y="length_plot", palette=colors, ax=axes[0],
                  inner="box", linewidth=1, cut=0)
    
    axes[0].set_xlabel("")
    axes[0].set_ylabel(length_label)
    axes[0].set_title("Length by Annotation Status")
    axes[0].yaxis.grid(True, linestyle='--', alpha=0.3)
    axes[0].set_axisbelow(True)
    
    # Add counts
    for i, status in enumerate(["Annotated", "Novel/Unannotated"]):
        n = (df["status"] == status).sum()
        pct = n / len(df) * 100
        axes[0].text(i, axes[0].get_ylim()[1] * 0.98, f'n={n:,}\n({pct:.1f}%)',
                    ha='center', va='top', fontsize=9, color='#444444')
    
    # Right: Pie chart of annotation status
    status_counts = df["status"].value_counts()
    axes[1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%',
               colors=[colors[s] for s in status_counts.index],
               explode=[0.02] * len(status_counts), startangle=90)
    axes[1].set_title("Annotation Coverage")
    
    fig.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    if show:
        plt.show()
    else:
        plt.close(fig)
    
    if return_data:
        return df, fig
    return None


In [ ]:
#| export

import numpy as np
import pysam
from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Union
import matplotlib.pyplot as plt
import sys
import time
from multiprocessing import Pool, cpu_count
from intervaltree import IntervalTree
import warnings
from typing import Optional, Union, List, Tuple, Dict
warnings.filterwarnings('ignore')


@dataclass
class CoverageResult:
    coverage: np.ndarray
    five_prime_bias: float
    three_prime_bias: float
    ratio: float
    n_transcripts: int
    n_bins: int
    total_records: int
    mapped_reads: int
    elapsed_seconds: float
    bam_path: str


def _process_chromosome(args):
    """Worker function to process a single chromosome - stores per-base coverage like Picard."""
    chrom, bam_path, exon_tuples, transcript_info, _ = args
    
    # Build interval tree for this chromosome
    tree = IntervalTree()
    for start, end, tx_id, exon_idx in exon_tuples:
        tree.addi(start, end, (tx_id, exon_idx))
    
    # Per-base coverage: tx_id -> array of length tx_length
    local_coverage = {}
    
    record_count = 0
    mapped_count = 0
    overlap_count = 0
    
    try:
        with pysam.AlignmentFile(bam_path, 'rb', threads=2) as bam:
            for read in bam.fetch(chrom):
                record_count += 1
                
                if read.is_unmapped or read.is_secondary or read.is_supplementary:
                    continue
                
                mapped_count += 1
                blocks = read.get_blocks()
                if not blocks:
                    continue
                
                for block_start, block_end in blocks:
                    overlapping = tree.overlap(block_start, block_end)
                    
                    for interval in overlapping:
                        exon_start, exon_end = interval.begin, interval.end
                        tx_id, exon_idx = interval.data
                        
                        ov_start = max(block_start, exon_start)
                        ov_end = min(block_end, exon_end)
                        
                        if ov_start >= ov_end:
                            continue
                        
                        overlap_count += 1
                        
                        tx_length, cum_lengths, strand = transcript_info[tx_id]
                        
                        # Initialize coverage array if needed
                        if tx_id not in local_coverage:
                            local_coverage[tx_id] = np.zeros(tx_length, dtype=np.int32)
                        
                        # Convert to transcript coordinates
                        tx_ov_start = cum_lengths[exon_idx] + (ov_start - exon_start)
                        tx_ov_end = cum_lengths[exon_idx] + (ov_end - exon_start)
                        
                        # Add coverage to each base (like Picard's addCoverageCounts)
                        local_coverage[tx_id][tx_ov_start:tx_ov_end] += 1
    
    except Exception as e:
        return chrom, {}, 0, 0, 0, str(e)
    
    return chrom, local_coverage, record_count, mapped_count, overlap_count, None


def compute_gene_body_coverage(
    bam_path: str,
    td,
    *,
    n_bins: int = 101,  # Picard uses 0-100 inclusive = 101 values
    min_length: int = 500,
    top_n: int = 1000,
    n_workers: int = 20,
    verbose: bool = True,
) -> CoverageResult:
    """Gene body coverage matching Picard's algorithm exactly."""
    
    start_time = time.time()
    
    if verbose:
        print("=" * 60)
        print("STEP 1: Building transcript database")
        sys.stdout.flush()
    
    transcripts = {}
    gene_transcripts = defaultdict(list)
    
    for tx_id, rec in td._idx._tx_by_id.items():
        if rec.exons is None or rec.exons.size == 0:
            continue
        exons = rec.exons[np.argsort(rec.exons[:, 0])].astype(np.int64)
        cum_lengths = np.zeros(len(exons), dtype=np.int64)
        running_sum = 0
        for i, (start, end) in enumerate(exons):
            cum_lengths[i] = running_sum
            running_sum += (end - start)
        tx_length = running_sum
        if tx_length < min_length:
            continue
        strand = 1 if rec.strand == 1 else -1
        gene_name = getattr(rec, 'gene_name', None) or getattr(rec, 'gene_id', tx_id)
        transcripts[tx_id] = {
            'exons': exons,
            'strand': strand,
            'chrom': rec.chrom,
            'length': tx_length,
            'cum_lengths': cum_lengths,
            'gene_name': gene_name,
        }
        gene_transcripts[gene_name].append(tx_id)
    
    if verbose:
        print(f"  {len(transcripts):,} transcripts from {len(gene_transcripts):,} genes")
        sys.stdout.flush()
    
    # Build per-chromosome exon data
    if verbose:
        print("=" * 60)
        print("STEP 2: Building per-chromosome exon index")
        sys.stdout.flush()
    
    chrom_exon_data = defaultdict(list)
    transcript_info = {}
    
    for tx_id, tx in transcripts.items():
        transcript_info[tx_id] = (tx['length'], list(tx['cum_lengths']), tx['strand'])
        for i, (start, end) in enumerate(tx['exons']):
            chrom_exon_data[tx['chrom']].append((int(start), int(end), tx_id, i))
    
    chromosomes = list(chrom_exon_data.keys())
    if verbose:
        print(f"  {len(chromosomes)} chromosomes with exon data")
        sys.stdout.flush()
    
    worker_args = [
        (chrom, bam_path, chrom_exon_data[chrom], transcript_info, n_bins)
        for chrom in chromosomes
    ]
    
    if verbose:
        print("=" * 60)
        print(f"STEP 3: Processing BAM with {n_workers} workers")
        print("-" * 60)
        sys.stdout.flush()
    
    # Process chromosomes in parallel
    total_records = 0
    total_mapped = 0
    total_overlaps = 0
    merged_coverage = {}  # tx_id -> per-base coverage array
    
    with Pool(n_workers) as pool:
        for i, result in enumerate(pool.imap_unordered(_process_chromosome, worker_args)):
            chrom, chrom_coverage, rec_count, map_count, ov_count, error = result
            
            if error:
                print(f"  ERROR on {chrom}: {error}")
                continue
            
            total_records += rec_count
            total_mapped += map_count
            total_overlaps += ov_count
            
            # Merge coverage arrays
            for tx_id, cov in chrom_coverage.items():
                if tx_id in merged_coverage:
                    merged_coverage[tx_id] += cov
                else:
                    merged_coverage[tx_id] = cov
            
            if verbose:
                elapsed = time.time() - start_time
                print(f"  [{i+1:>2}/{len(chromosomes)}] {chrom:>5}: {rec_count:>10,} records | {elapsed/60:.1f}m")
                sys.stdout.flush()
    
    elapsed = time.time() - start_time
    if verbose:
        print("-" * 60)
        print(f"  Done: {total_records:,} records, {total_mapped:,} mapped")
        print(f"  Time: {elapsed/60:.1f} minutes")
        sys.stdout.flush()
    
    # Select best transcript per gene (like Picard's pickTranscripts)
    if verbose:
        print("=" * 60)
        print("STEP 4: Selecting best transcript per gene")
        sys.stdout.flush()
    
    best_per_gene = {}
    for gene_name, tx_list in gene_transcripts.items():
        best_tx = None
        best_mean = 0
        
        for tx_id in tx_list:
            if tx_id not in merged_coverage:
                continue
            cov = merged_coverage[tx_id]
            mean_cov = cov.mean()
            if mean_cov < 1.0:  # Picard: if (mean < 1d) continue
                continue
            if best_tx is None or mean_cov > best_mean:
                best_tx = tx_id
                best_mean = mean_cov
        
        if best_tx is not None:
            best_per_gene[gene_name] = (best_tx, best_mean)
    
    # Take top 1000 by coverage
    sorted_genes = sorted(best_per_gene.items(), key=lambda x: x[1][1], reverse=True)[:top_n]
    
    if verbose:
        print(f"  Selected {len(sorted_genes)} genes with coverage >= 1")
        sys.stdout.flush()
    
    # Compute normalized coverage using Picard's exact method
    if verbose:
        print("=" * 60)
        print("STEP 5: Computing normalized coverage (Picard method)")
        sys.stdout.flush()
    
    # Picard uses 101 bins (0-100 inclusive)
    normalized_by_position = np.zeros(101, dtype=np.float64)
    transcript_count = len(sorted_genes)
    
    for gene_name, (tx_id, _) in sorted_genes:
        cov = merged_coverage[tx_id].astype(np.float64)
        strand = transcripts[tx_id]['strand']
        
        # Reverse for negative strand (like Picard)
        if strand == -1:
            cov = cov[::-1]
        
        mean_cov = cov.mean()
        if mean_cov == 0:
            continue
        
        last_index = len(cov) - 1
        
        # Picard's exact binning: ±0.5% window around each percentile
        for percent in range(101):
            p = percent / 100.0
            start = int(max(0, last_index * (p - 0.005)))
            end = int(min(last_index, last_index * (p + 0.005)))
            length = end - start + 1
            
            window_sum = cov[start:end+1].sum()
            normalized = (window_sum / length) / mean_cov
            normalized_by_position[percent] += normalized / transcript_count
    
    # Compute bias metrics (Picard uses first/last 100 bases, we approximate with bins)
    five_prime = normalized_by_position[:10].mean()
    three_prime = normalized_by_position[-10:].mean()
    ratio = three_prime / five_prime if five_prime > 0 else 0
    
    elapsed = time.time() - start_time
    if verbose:
        print("=" * 60)
        print(f"COMPLETE in {elapsed/60:.1f} minutes")
        print(f"  5' bias: {five_prime:.3f}, 3' bias: {three_prime:.3f}, ratio: {ratio:.2f}")
        sys.stdout.flush()
    
    return CoverageResult(
        coverage=normalized_by_position,
        five_prime_bias=five_prime,
        three_prime_bias=three_prime,
        ratio=ratio,
        n_transcripts=len(sorted_genes),
        n_bins=101,
        total_records=total_records,
        mapped_reads=total_mapped,
        elapsed_seconds=elapsed,
        bam_path=bam_path,
    )


def plot_gene_body_coverage(
    results: Union[CoverageResult, List[CoverageResult]],
    labels: List[str] = None,
    title: str = "Gene Body Coverage",
    figsize: tuple = (10, 6),
    normalize: str = "max",  # "max" (peak=1.0) or "mean" (Picard-style, values ~1.0)
) -> plt.Figure:
    """Plot gene body coverage from one or more CoverageResult objects."""
    
    if isinstance(results, CoverageResult):
        results = [results]
    
    if labels is None:
        labels = [r.bam_path.split('/')[-1] for r in results]
    
    fig, ax = plt.subplots(figsize=figsize)
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
    
    for result, label, color in zip(results, labels, colors):
        x = np.arange(result.n_bins)  # 0-100
        cov = result.coverage.copy()
        if normalize == "max":
            cov = cov / cov.max()
        ax.plot(x, cov, label=f"{label} (3'/5'={result.ratio:.2f})", color=color, linewidth=2)
    
    ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel("Normalized Position Along Transcript (5' -> 3')")
    ax.set_ylabel("Normalized Coverage")
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(0, 100)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig


## Fast Gene Body Coverage (Picard-speed)

This implementation uses a **single-pass algorithm** matching Picard's performance:
- Reads the BAM file only **once** (sequential I/O)
- Uses **interval trees** for O(log n) transcript lookups per read
- ~30-40 minutes vs hours for the multi-query approach

**Requirements:** `pip install intervaltree`


## Gene Body Coverage QC (Picard-style + Long-read Extensions)

## Long-Read Single-Cell Specific QC

## Transcript Length Analysis

In [ ]:
#| export
from typing import Optional, Tuple
def plot_transcript_length_abundance(
    td,
    adata,
    *,
    adata_tx_col: Optional[str] = None,
    bins: int = 60,
    y_axis: str = "percent",  # "percent", "count", or "density"
    title: str = "Transcript Length vs Abundance",
    figsize: Tuple[float, float] = (8, 5),
    dpi: int = 150,
    save: Optional[str] = None,
):
    from scipy.stats import gaussian_kde
    import seaborn as sns
    _setup_scanpy_style()

    length_table = td.transcript_lengths_table()

    if adata_tx_col is None:
        tx_ids = _strip_ver(adata.var.index)
    else:
        tx_ids = _strip_ver(adata.var[adata_tx_col].values)

    counts = np.array(adata.X.sum(axis=0)).flatten()
    count_df = pd.DataFrame({"transcript_id": tx_ids, "total_counts": counts})

    df = length_table.merge(count_df, on="transcript_id", how="inner")
    df = df[(df["length"] > 0) & (df["total_counts"] > 0)]

    if df.empty:
        print("No matching transcripts found.")
        return

    lengths_expanded = np.repeat(df["length"].values, df["total_counts"].values.astype(int))
    log_lengths = np.log10(lengths_expanded)
    w_median_log = np.median(log_lengths)
    w_median_bp = 10 ** w_median_log

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)

    if y_axis == "percent":
        weights = np.ones(len(log_lengths)) / len(log_lengths) * 100
        ylabel = "% of UMIs"
        hist_kwargs = dict(weights=weights, density=False)
    elif y_axis == "count":
        ylabel = "Number of UMIs"
        hist_kwargs = dict(density=False)
    elif y_axis == "density":
        ylabel = "Density"
        hist_kwargs = dict(density=True)
    else:
        raise ValueError("`y_axis` must be 'percent', 'count', or 'density'")

    ax.hist(log_lengths, bins=bins,
            color=ALLOS_PALETTE[0], edgecolor="white", linewidth=0.3, alpha=0.75,
            **hist_kwargs)

    # Smooth KDE scaled to match y-axis
    kde = gaussian_kde(log_lengths, bw_method=0.15)
    x_range = np.linspace(log_lengths.min(), log_lengths.max(), 500)
    bin_width = (log_lengths.max() - log_lengths.min()) / bins
    if y_axis == "percent":
        kde_y = kde(x_range) * bin_width * 100
    elif y_axis == "count":
        kde_y = kde(x_range) * bin_width * len(log_lengths)
    else:
        kde_y = kde(x_range)
    ax.plot(x_range, kde_y, color="white", lw=2.5)
    ax.plot(x_range, kde_y, color=ALLOS_PALETTE[0], lw=1.5, alpha=0.9)

    ax.axvline(w_median_log, color="#e05c2a", linestyle="--", lw=1.5, zorder=5)
    ax.text(w_median_log - 0.03, ax.get_ylim()[1] * 0.92,
            f"weighted median\n{w_median_bp:,.0f} bp",
            fontsize=8, color="#e05c2a", va="top", ha="right")

    for bp in [500, 1000, 2000, 5000, 10000]:
        lv = np.log10(bp)
        if ax.get_xlim()[0] < lv < ax.get_xlim()[1]:
            ax.axvline(lv, color="gray", linestyle=":", alpha=0.4, linewidth=1)

    tick_bps = [100, 500, 1000, 2000, 5000, 10000, 50000, 100000]
    ax.set_xticks([np.log10(b) for b in tick_bps])
    ax.set_xticklabels(["100bp", "500bp", "1kb", "2kb", "5kb", "10kb", "50kb", "100kb"],
                       rotation=30, ha="right", fontsize=8)
    ax.set_xlim(log_lengths.min() - 0.1, log_lengths.max() + 0.1)

    stats_text = (f"Total UMIs: {len(lengths_expanded):,}\n"
                  f"Transcripts detected: {len(df):,}")
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9, edgecolor="#cccccc"))

    ax.set_xlabel("Transcript length")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=15)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()

    if save:
        fig.savefig(save, bbox_inches="tight", dpi=dpi)
    plt.show()


### Length Bias QC

### Biotype Analysis

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()